**Runs on Kaggle: [kaggle.com/code/jhffmn/threadatlas-global-layer](https://www.kaggle.com/code/jhffmn/threadatlas-global-layer)**

This notebook is ThreadAtlas's the global layer. It runs on Kaggle with the inputs attached there; this copy is the version that made its output, [the store](https://www.kaggle.com/datasets/jhffmn/threadatlas-global-chunks).

# ThreadAtlas global layer

This notebook reads a mixed corpus of chat sessions, books and papers into one database. Every
document keeps its own entities, facts, narratives and abstracts. A **parent** sits over the
entities that are the same person, place or thing in different documents, so they are connected
while every claim stays with the document that made it. Nothing is merged. A parent asserts nothing
about the world: it holds a name, a kind, its children's aliases and one line per instance.

A document with a real title is a parent too. Its own node is placed like any other entity, so a
book mentioned in another document ends up under the same parent as the book itself.

The corpus arrives in ten periodic dumps, each a mix of everything and each read in date order.
There is one graph over all of it. A test that needs one user's history filters at evaluation, and
the same graph is also measured with no filter.

## What comes in

Two public datasets. [ThreadAtlas Step 1](https://www.kaggle.com/datasets/jhffmn/it494-threadatlas-step1)
has one JSON Lines file per record type, every row keyed by `doc_id`.
[ThreadAtlas Step 0](https://www.kaggle.com/datasets/jhffmn/it494-threadatlas-step0) holds the
documents' text, which Step 1 does not carry. Every quote is checked against it at load.

## What goes out

| file | what it holds |
|---|---|
| `threadatlas.sqlite` | the serving store |
| `threadatlas.npy` | one vector per row of `vec_row` |
| `build.sqlite` | every pair put to the judge, with its signals, verdict and reason |
| `decisions.jsonl` | where each entity went; a stopped run replays this |
| `parents.jsonl` | every parent written by a model call |
| `library-groups.json` | the bodies of work |
| `dumps.jsonl` | the size of the graph and the spend after each dump |
| `calls.jsonl` | every model call and its cost |
| `receipt.json`, `oz-score.json`, `joins-to-read.md`, `collection-radial.svg` | the counts, the Oz score, the sample of joins for a person to read, the figure |

Retrieval is its own pipeline over the store (`docs/retrieval.md`).

## The algorithm

```
load the store: every Step 1 record, each quote checked against its document's text

cut the corpus into 10 dumps:
    for each source folder:
        sort its documents by date, cut them into 10 equal runs
    dump k = run k of every folder, sorted by date

for each dump, in order:
    embed this dump's facts, narrative sentences and entities

    for each document in the dump, started in date order, 32 side by side:
        for each entity in the document, its own titled node included:
            offer = parents sharing a name with it, closest first
                    then parents with a placed entity at cosine .80 or more, closest first
                    keep the best 5
        up to 5 rounds:
            judge each unplaced entity against its next offered parent, all at once
            "same" attaches it
        judge each is_a link inside the document that the rounds left apart
        an entity with no "same" founds a new parent
        log the document, so a stopped run replays the log and goes on

    join:     each changed parent against up to 5 parents sharing a name, each pair judged once
    rewrite:  each parent that changed gets one call, which picks its name and writes its lines
    group:    each new book or paper is placed into a body of work the same way, by shared
              entities and abstract similarity; changed bodies are renamed and given subgroups
              after the last dump, every group is compared once more

one parent per speaker over the `user` of each of their conversations, with no call
embed the parents' summaries into the sidecar
cut every document into the 256-token chunks plain RAG uses: offsets, keyword rows, vectors, and the
    facts each chunk holds
group the chat sessions by topic
write the Oz key's score, the sample of joins to read, and the figure
```

Documents running side by side do not see each other, so they enter the graph slightly out of
order. Two that hold the same entity found two parents, and the join at the end of the dump unites
them. A document's node and an entity of the same name inside it are united the same way.

## The store

The serving store keeps what a question reads. The document's text stays in Step 0. The build's
evidence goes to `build.sqlite` beside it.

| table | columns | meaning |
|---|---|---|
| `document` | doc_id, title, author, occurred_at, source_uri | the filter's unit; the date on every item |
| `unit` | unit_id, doc_id, position, label, kind, occurred_at | order inside a document; a chapter's or a turn's label |
| `node` | doc_id, node_id, name, kind | a document's entity |
| `alias` | doc_id, node_id, alias | its surface forms |
| `fact` | doc_id, fact_id, subject, subject_name, predicate, object, object_is_node, qualifiers, unit_id, quote, quote_start, quote_end, occurred_at, direction | a claim with its verbatim quote |
| `adjudicated_fact` | doc_id, node_id, predicate, object, qualifiers, from_facts | the ingestor's consolidation |
| `cell`, `abstract` | doc_id, node_id, unit_id, text; doc_id, node_id, text | the narratives |
| `parent` | parent_id, name, kind, aliases, summary | a global entity: derived fields only |
| `instance_of` | doc_id, node_id, parent_id | the up-edge, owned by the document |
| `collection`, `document_in` | collection_id, name, abstract, within; doc_id, collection_id | a body of work, a subgroup inside one (`within`), or a chat topic, and its documents |
| `chunk` | doc_id, chunk_id, start, end | a 256-token stretch of the document's text, by its offsets |
| `fact_in_chunk` | doc_id, fact_id, chunk_id | the chunks that hold a fact's whole quote |
| `vec_header`, `vec_row` | model, dimension; row, record, doc_id, record_id | what each row of the array is |
| `search` | record, doc_id, record_id, text (FTS5) | one row per record: a fact's line and quote, a cell, an abstract, a chunk's text |

## The sidecar

One float16 vector per row of `vec_row`: every fact as a line (subject, predicate, object,
qualifiers, date, document), every sentence of every cell, abstract and parent summary, prefixed
with its entity and, for a book or paper, its title, and every chunk as it stands. `bge-small-en-v1.5` through fastembed, 384
dimensions. Search is a linear scan of the array.

## Measuring the signals

With `SHOW_CAST` on, the judge sees the names each entity appears beside, those the two share first.
With it off, the judge sees the two entities' texts alone. That is the attachment ablation.

## Running it

On Kaggle: attach the two datasets and the Oz key, add the `OPENAI_API_KEY` secret, turn Internet on,
and run all. A session stops itself after `TIME_STOP_HOURS`. To go on, turn that session's output
into a dataset, attach it, and run again: the notebook copies it back and carries on from the last
logged document. Locally: `pip install bertopic fastembed numpy matplotlib`, set `STEP1`, `EXPORT`,
`OUT` and `OPENAI_API_KEY`, and run the script.

## Block 1: files and settings

Where the inputs are, where the outputs go, and the constants.

```
find the inputs (Kaggle's mount, or the environment)
if an earlier session's output is attached, copy it into the working folder
```

In [ ]:
import json
import math
import os
import random
import shutil
import sqlite3
import sys
import time
from pathlib import Path, PurePosixPath

VERSION = "threadatlas-global-layer 0.7"

VECTOR_FLOOR = 0.80        # cosine at which a placed entity's parent is offered with no shared name
TRIES = 5                  # parents an entity may be judged against; 98.6% of matches came within five
DUMPS = 10                 # the corpus arrives as this many dumps of equal size
FACTS_SHOWN = 12           # facts of an entity shown to the judge and the parent writer
SIDE_SHOWN = 6             # instances shown per side to the judge
GROUP_SHOWN = 20           # instances shown to the call that writes a parent, most facts first
TITLES_SHOWN = 30          # titles of a body of work shown to the judge
NAMES_SHOWN = 8            # shared entities shown to the judge of a body of work
RRF_K = 60                 # reciprocal rank fusion, the same as retrieval
WORKERS = 32               # documents side by side, and calls in flight at once
SHOW_CAST = True           # False is the ablation's arm with no co-occurrence
SPEND_STOP = 150.0         # dollars; the run stops here and can be resumed
TIME_STOP_HOURS = 8        # a session stops itself here, between steps
FLAT_TOPICS = 16           # past this many chat topics a reader needs a level above them
TOPIC_FLOOR = 0.65         # a chat session under this cosine to every topic centroid is Other

# the embedder and BERTopic are not on Kaggle's image; fetched once, with Internet on
try:
    import fastembed
    import bertopic
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastembed", "bertopic"], check=True)
import numpy


def first_existing(variable, folders):
    """The folder the environment names, else the first of `folders` that exists, else None."""
    if os.environ.get(variable):
        return Path(os.environ[variable])
    for folder in folders:
        if folder.exists():
            return folder
    return None


STEP1 = first_existing("STEP1", [Path("/kaggle/input/datasets/jhffmn/it494-threadatlas-step1"),
                                 Path("/kaggle/input/it494-threadatlas-step1")])
EXPORT = first_existing("EXPORT", [Path("/kaggle/input/datasets/jhffmn/it494-threadatlas-step0"),
                                   Path("/kaggle/input/it494-threadatlas-step0")])
OZ_KEY = first_existing("OZ_KEY", [Path("/kaggle/input/datasets/jhffmn/it494-threadatlas-oz-key"),
                                   Path("/kaggle/input/it494-threadatlas-oz-key")])
OUT = Path(os.environ.get("OUT", "/kaggle/working"))
OUT.mkdir(parents=True, exist_ok=True)
STORE = OUT / "threadatlas.sqlite"
DECISIONS = OUT / "decisions.jsonl"
WRITTEN_PARENTS = OUT / "parents.jsonl"
LIBRARY = OUT / "library-groups.json"
STARTED = time.time()


def restore_run():
    """Kaggle's working folder starts empty, so an attached earlier session is copied back."""
    if DECISIONS.exists():
        return
    earlier = None
    longest = -1
    for path in Path("/kaggle/input").rglob("decisions.jsonl"):       # the longest log is the latest
        size = path.stat().st_size
        if size > longest:
            earlier = path.parent
            longest = size
    if earlier is not None:
        shutil.copytree(earlier, OUT, dirs_exist_ok=True)
        print(f"going on from {earlier}")


restore_run()
print(f"{VERSION}\nStep 1: {STEP1}\nStep 0: {EXPORT}\noutput: {OUT}")


def read_jsonl(path):
    """One dict per line. A session Kaggle kills can leave a line cut short; that line is skipped."""
    rows = []
    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except ValueError:
                if line.strip():
                    print(f"skipped a cut line in {Path(path).name}")
    return rows


def fold(text):
    """Lowercase words joined by single spaces: the form two names are matched in."""
    words = []
    word = ""
    for c in (text or "").casefold():
        if "a" <= c <= "z" or "0" <= c <= "9":
            word += c
        elif word:
            words.append(word)
            word = ""
    if word:
        words.append(word)
    return " ".join(words)


def is_identifier(title, source_uri):
    """A chat has no title of its own, and a file's own name is not a title either."""
    return title is None or Path(source_uri).name.startswith(title + ".")


def document_name(doc):
    """What a document is called in a parent's lines: its title, or what it is and its date when it has none."""
    if not is_identifier(doc["title"], doc["source_uri"]):
        return doc["title"]
    what = "an untitled work"
    if doc["chat"]:
        what = "a conversation"
    return f"{what} of {(doc['occurred_at'] or 'unknown date')[:10]}"

## Block 2: the model

`generate` is the only way this notebook calls a model: raw HTTP and JSON replies. Every call goes to
`calls.jsonl` with its model, tier, tokens, seconds and cost.

```
generate(prompt, required shape):
    up to 2 times:
        stop the run if the spend has reached SPEND_STOP
        post the prompt (3 attempts on a timeout or a busy server; flex full -> standard tier)
        if that fails, return nothing
        log the call and its cost
        if the reply fits the shape, return it
        otherwise append the error to the prompt
    return nothing
```

In [ ]:
import http.client
import threading
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

LUNA = "gpt-5.6-luna"                   # the parent writer, and the judge when a chat is on either side
TERRA = "gpt-5.6-terra"                 # the judge for pairs of book and paper children
SERVICE_TIER = "flex"
PRICE = {"flex": {LUNA: (0.10, 0.60), TERRA: (1.00, 6.00)},          # $ per million tokens in, out
         "default": {LUNA: (0.20, 1.20), TERRA: (2.00, 12.00)}}
CALLS = OUT / "calls.jsonl"
LOG_LOCK = threading.Lock()

SPENT = 0.0                             # what this run has spent, earlier sessions included
if CALLS.exists():
    for row in read_jsonl(CALLS):
        SPENT += row["cost"]

KEY = os.environ.get("OPENAI_API_KEY")
if not KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
    except Exception:
        KEY = None


class StopRun(Exception):
    """The session stops here, on time or on spend, and the next one goes on."""


def check_time():
    """Past TIME_STOP_HOURS a session stops between steps, in time to save its output."""
    if time.time() - STARTED > TIME_STOP_HOURS * 3600:
        raise StopRun(f"time stop after {TIME_STOP_HOURS} hours")


def post(payload):
    data = json.dumps(payload).encode("utf-8")
    headers = {"Authorization": f"Bearer {KEY}", "Content-Type": "application/json"}
    request = urllib.request.Request("https://api.openai.com/v1/chat/completions", data=data, method="POST", headers=headers)
    with urllib.request.urlopen(request, timeout=900) as response:
        return response.status, response.read().decode("utf-8")


def log_call(body, model, stage, seconds):
    """Write one call to calls.jsonl and add its cost to SPENT."""
    global SPENT
    tier = body.get("service_tier") or SERVICE_TIER
    price_in, price_out = PRICE.get(tier, PRICE["default"])[model]
    usage = body.get("usage", {})
    tokens_in = usage.get("prompt_tokens", 0)
    tokens_out = usage.get("completion_tokens", 0)
    cost = (tokens_in * price_in + tokens_out * price_out) / 1e6
    row = {"stage": stage, "model": body.get("model", model), "tier": tier, "in": tokens_in, "out": tokens_out,
           "seconds": seconds, "cost": cost}
    with LOG_LOCK:
        SPENT += cost
        with CALLS.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def call(payload, model, stage):
    """One exchange with up to 3 attempts; the parsed body."""
    if SPENT >= SPEND_STOP:
        raise StopRun(f"spending stop: ${SPENT:.2f} of ${SPEND_STOP:.2f}")
    started = time.time()
    status = None
    text = ""
    for attempt in range(3):
        try:
            status, text = post(payload)
        except urllib.error.HTTPError as error:
            status = error.code
            text = error.read().decode("utf-8", errors="replace")
        except (OSError, http.client.HTTPException) as error:
            status = None
            text = str(error)
        if status == 200:
            break
        if status == 429 and "insufficient_quota" in text:
            raise StopRun(f"OpenAI quota exhausted: {text[:200]}")
        if status == 429 and "flex_unavailable" in text:       # flex has no room for this call
            payload = dict(payload)
            payload["service_tier"] = "default"
            continue
        if status is not None and status != 429 and status < 500:
            raise RuntimeError(f"OpenAI {status}: {text}")
        time.sleep(15 * (attempt + 1))
    if status != 200:
        raise RuntimeError(f"OpenAI {status}: {text}")
    body = json.loads(text)
    log_call(body, model, stage, round(time.time() - started, 1))
    return body


def fits(reply, schema):
    """`schema` maps each required key to its type, or None for any. The first problem, or None."""
    if not isinstance(reply, dict):
        return "reply is not an object"
    for key, kind in schema.items():
        if key not in reply:
            return f"missing {key}"
        if kind is not None and not isinstance(reply[key], kind):
            return f"{key} is not {kind.__name__}"
    return None


def generate(prompt, schema, stage, model=LUNA, effort="low"):
    """The reply as a dict that fits `schema`, or None if the call fails or two replies do not fit."""
    for attempt in range(2):
        payload = {"model": model, "reasoning_effort": effort, "service_tier": SERVICE_TIER,
                   "response_format": {"type": "json_object"}, "messages": [{"role": "user", "content": prompt}]}
        try:
            body = call(payload, model, stage)
        except (RuntimeError, ValueError) as error:
            print(f"  no reply at {stage}: {str(error)[:120]}")
            return None
        try:
            reply = json.loads(body["choices"][0]["message"]["content"] or "")
            problem = fits(reply, schema)
        except (ValueError, KeyError, IndexError, TypeError) as error:
            problem = f"{type(error).__name__}: {error}"
        if problem is None:
            return reply
        prompt += f"\n\nYour previous reply did not fit the required shape ({problem}). "
        prompt += "Reply again, in exactly the shape asked for."
    return None


CALL_POOL = None


def in_parallel(function, jobs):
    """function(*job) for every job, results in the jobs' order. One pool serves the whole run."""
    global CALL_POOL
    if CALL_POOL is None:
        CALL_POOL = ThreadPoolExecutor(max_workers=WORKERS)
    futures = []
    for job in jobs:
        futures.append(CALL_POOL.submit(function, *job))
    results = []
    for future in futures:
        results.append(future.result())
    return results

## Block 3: connection test

One tiny call, made before anything spends.

```
ask for {"ok": true}; stop if it does not come back
```

In [ ]:
if __name__ == "__main__":
    if not generate('Reply with the JSON object {"ok": true}.', {"ok": None}, "ping"):
        raise SystemExit("the connection test failed: check OPENAI_API_KEY and that Internet is on")

## Block 4: the store's schema

The serving store holds what a question reads: fourteen tables and one full-text table with a row per
record. The judged pairs go to `build.sqlite`.

```
serving store: document, unit, node, alias, fact, adjudicated_fact, cell, abstract,
               parent, instance_of, collection, document_in, vec_header, vec_row, search
build log:     pair
```

In [ ]:
SCHEMA = """
create table document (doc_id text primary key, title text, author text, occurred_at text, source_uri text);
create table unit (unit_id text primary key, doc_id text, position integer, label text, kind text, occurred_at text);
create table node (doc_id text, node_id text, name text, kind text, primary key (doc_id, node_id));
create table alias (doc_id text, node_id text, alias text);
create table fact (doc_id text, fact_id text, subject text, subject_name text, predicate text, object text,
    object_is_node integer, qualifiers text, unit_id text, quote text, quote_start integer, quote_end integer,
    occurred_at text, direction text, primary key (doc_id, fact_id));
create table adjudicated_fact (doc_id text, node_id text, predicate text, object text, qualifiers text, from_facts text);
create table cell (doc_id text, node_id text, unit_id text, text text);
create table abstract (doc_id text, node_id text, text text);

create table parent (parent_id integer primary key, name text, kind text, aliases text, summary text);
create table instance_of (doc_id text, node_id text, parent_id integer, primary key (doc_id, node_id));
create table collection (collection_id integer primary key, name text, abstract text, within integer);
create table document_in (doc_id text, collection_id integer, primary key (doc_id, collection_id));

create table vec_header (model text, dimension integer);
create table vec_row (row integer primary key, record text, doc_id text, record_id text);
create virtual table search using fts5(record unindexed, doc_id unindexed, record_id unindexed, text);
"""

BUILD_SCHEMA = """
create table pair (doc_a text, node_a text, doc_b text, node_b text, lexical real, vector real, cast real,
    cast_rare real, identity integer, verdict text, reason text);
"""

# the columns that get read; without these a lookup is a scan of a million rows
INDEXES = """
create index unit_doc on unit (doc_id);
create index alias_node on alias (doc_id, node_id);
create index fact_subject on fact (doc_id, subject);
create index adjudicated_node on adjudicated_fact (doc_id, node_id);
create index cell_node on cell (doc_id, node_id);
create index abstract_node on abstract (doc_id, node_id);
create index instance_parent on instance_of (parent_id);
create index member_collection on document_in (collection_id);
"""

CHATS = "select distinct doc_id from unit where kind in ('user', 'assistant')"    # a chat is a document of turns

## Block 5: loading the store

The Step 1 rows go in with the columns the store keeps. Every quote must slice from its document's
text at its offsets, and the load stops on the first mismatch. A document-record fact has no quote
and is not checked.

Step 1's output carries LongMemEval's marks from the benchmark's ids: `answer_` on a gold session and
`_abs` on an unanswerable question. The load removes them. Every chat loses its title, which was only
its id, and the id goes plain in the fields it was copied into. Only those exact fields are changed.
The source path keeps the id, which is how a record traces to its session and how the benchmark finds
its gold sessions.

```
for each table: insert the Step 1 rows it keeps, a document's own node last
for each fact with a quote:
    if text[quote_start:quote_end] != quote: stop the load
clear the marks: no chat keeps a title; the ids in labels, names and record facts go plain
add one search row per fact, cell and abstract
```

In [ ]:
# the Step 1 file behind each table, and the fields the store keeps
ROWS = {
    "document": ("documents", ["doc_id", "title", "author", "occurred_at", "source_uri"]),
    "unit": ("units", ["unit_id", "doc_id", "position", "label", "kind", "occurred_at"]),
    "node": ("nodes", ["doc_id", "node_id", "name", "kind"]),
    "alias": ("aliases", ["doc_id", "node_id", "alias"]),
    "adjudicated_fact": ("adjudicated_facts", ["doc_id", "node_id", "predicate", "object", "qualifiers", "from_facts"]),
    "cell": ("cells", ["doc_id", "node_id", "unit_id", "text"]),
    "abstract": ("abstracts", ["doc_id", "node_id", "text"]),
}


def column(value):
    """A value as SQLite stores it: nested values as JSON text."""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def texts_of(export_documents, doc_ids):
    """doc_id -> text from Step 0, used at load and not kept."""
    texts = {}
    with Path(export_documents).open(encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row["doc_id"] in doc_ids:
                texts[row["doc_id"]] = row["text"]
    return texts


def load_table(db, table, file, fields, step1):
    rows = []
    for record in read_jsonl(step1 / f"{file}.jsonl"):
        row = []
        for field in fields:
            row.append(column(record.get(field)))
        rows.append(row)
    if table == "document":                # a document with no title goes by its file name
        for row in rows:
            if not row[1]:
                row[1] = Path(row[4]).stem
    if table == "node":                    # retrieval takes the last node of kind `document`, so that must be the document's own
        entities = []
        document_nodes = []
        for row in rows:
            if row[1].endswith(":doc"):
                document_nodes.append(row)
            else:
                entities.append(row)
        rows = entities + document_nodes
    marks = ", ".join("?" * len(fields))
    db.executemany(f"insert into {table} values ({marks})", rows)


def load_facts(db, step1, texts):
    """Every fact, its quote checked against the text; how many were checked."""
    checked = 0
    for r in read_jsonl(step1 / "facts.jsonl"):
        if r["quote"] is not None:
            if texts[r["doc_id"]][r["quote_start"]:r["quote_end"]] != r["quote"]:
                raise SystemExit(f"quote of {r['fact_id']} does not slice to its text; load refused")
            checked += 1
        db.execute("insert into fact values (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                   (r["doc_id"], r["fact_id"], r["subject"], r["provenance"].get("subject_name"), r["predicate"], r["object"],
                    int(r["object_is_node"]), r["qualifiers"], r["unit_id"], r["quote"], r["quote_start"], r["quote_end"],
                    r["occurred_at"], r["direction"]))
    return checked


def plain_column(db, table, name, where):
    """One column's session ids made plain, in the rows named."""
    rows = []
    for rowid, value in db.execute(f"select rowid, {name} from {table} where {where}"):
        clean = plain(value + " ").strip()
        if clean != value:
            rows.append((clean, rowid))
    db.executemany(f"update {table} set {name} = ? where rowid = ?", rows)
    return len(rows)


def clear_marks(db):
    """LongMemEval's marks out of Step 1's output, before anything reads it; the source path keeps them."""
    chats = f"doc_id in ({CHATS})"
    db.execute(f"update document set title = null where {chats}")
    cleared = plain_column(db, "node", "name", f"node_id like '%:doc' and {chats}")
    cleared += plain_column(db, "unit", "label", chats)
    cleared += plain_column(db, "fact", "object", f"quote is null and predicate in ('has_title', 'belongs_to_history') and {chats}")
    cleared += plain_column(db, "fact", "subject_name", f"quote is null and {chats}")
    print(f"marks cleared from {cleared} fields")


def load_search(db):
    """One search row per record: a fact's line and quote, a cell's text, an abstract's text."""
    rows = []
    for doc_id, fact_id, line, quote in fact_lines(db):
        if quote:
            line += "\n" + quote
        rows.append(("fact", doc_id, fact_id, line))
    for doc_id, node_id, unit_id, text in db.execute("select doc_id, node_id, unit_id, text from cell"):
        rows.append(("cell", doc_id, f"{node_id}@{unit_id}", text))
    for doc_id, node_id, text in db.execute("select doc_id, node_id, text from abstract"):
        rows.append(("abstract", doc_id, node_id, text))
    db.executemany("insert into search values (?,?,?,?)", rows)
    return len(rows)


def build_store(step1, export_documents, path):
    """The serving store from Step 1 and Step 0's documents.jsonl, with build.sqlite beside it."""
    db = sqlite3.connect(path)
    db.executescript(SCHEMA)
    log = sqlite3.connect(path.with_name("build.sqlite"))
    log.executescript(BUILD_SCHEMA)
    log.close()
    for table, (file, fields) in ROWS.items():
        load_table(db, table, file, fields, step1)
    doc_ids = set()
    for (doc_id,) in db.execute("select doc_id from document"):
        doc_ids.add(doc_id)
    texts = texts_of(export_documents, doc_ids)
    missing = doc_ids - set(texts)
    if missing:
        raise SystemExit(f"{len(missing)} documents have no text in Step 0, first {sorted(missing)[0][:8]}")
    checked = load_facts(db, step1, texts)
    clear_marks(db)
    searchable = load_search(db)
    db.executescript(INDEXES)
    db.commit()
    db.close()
    print(f"store: {checked} quotes checked, all slice to their text; {searchable} search rows")

## Block 6: what gets a vector

The texts of the sidecar: every fact as one line, every sentence of every cell and abstract with its
entity in front (and the title, when the document has one), and every sentence of every parent
summary with the parent's name in front.

```
fact      -> "subject predicate object (qualifiers), date, title"
cell      -> one row per sentence: "entity, title: sentence"
abstract  -> one row per sentence: "entity, title: sentence"
parent    -> one row per sentence, parents of 2 or more only: "name: sentence"
```

A fact marked `inverse` was read with a minor as its subject and stored under the major it points
at, so its line is written the way it was read: the object first. The texts here carry neither
LongMemEval mark, `answer_` or `_abs`. The store keeps the ids; only what is searched is plain.

In [ ]:
MODEL = "BAAI/bge-small-en-v1.5"
DIMENSION = 384
SENTENCE_OPENERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789\"'(["


def sentences(text):
    """Cut text after . ! or ? where white space and a sentence opener follow."""
    text = (text or "").strip()
    pieces = []
    start = 0
    i = 1
    while i < len(text):
        if text[i - 1] in ".!?" and text[i].isspace():
            opener = i
            while opener < len(text) and text[opener].isspace():
                opener += 1
            if opener < len(text) and text[opener] in SENTENCE_OPENERS:
                pieces.append(text[start:i])
                start = opener
            i = opener
        i += 1
    pieces.append(text[start:])
    kept = []
    for piece in pieces:
        if piece.strip():
            kept.append(piece)
    return kept


def plain(text):
    """A text without the marks LongMemEval puts in session ids: answer_ on gold, _abs on an unanswerable history's."""
    text = text.replace("answer_", "")
    for end in ("_", " ", ":", ",", ".", ")", "\n"):           # the end of an id, so a book's _absolute_ stays
        text = text.replace("_abs" + end, end)
    return text


def fact_text(f, names):
    """A fact as a clause: subject, predicate, object, qualifiers."""
    subject = names.get(f["subject"], f["subject"])
    if f["direction"] == "mentioned" and f["subject_name"]:    # a chat minor's fact, carried by its session
        subject = f["subject_name"]
    obj = f["object"]
    if f["object_is_node"]:
        obj = names.get(f["object"], f["object"])
    if f["predicate"] == "belongs_to_history":                 # a history's folder is its question's id
        obj = obj.removesuffix("_abs")
    if f["direction"] == "inverse":                            # stored under the major its object points at
        subject, obj = obj, subject
    line = f"{subject} {f['predicate'].replace('_', ' ')} {obj}"
    if f["qualifiers"]:
        line += f" ({f['qualifiers']})"
    return line


def fact_line(f, names, doc):
    """The clause with its date and its document's title: the sentence a question would match."""
    line = fact_text(f, names)
    if f["occurred_at"]:
        line += f", {f['occurred_at']}"
    if not is_identifier(doc["title"], doc["source_uri"]):
        line += f", {doc['title']}"
    return plain(line)


def prefix(entity, doc):
    if is_identifier(doc["title"], doc["source_uri"]):
        return plain(f"{entity}: ")
    return f"{entity}, {doc['title']}: "


def read_titles(db):
    docs = {}
    for doc_id, title, source_uri in db.execute("select doc_id, title, source_uri from document"):
        docs[doc_id] = {"title": title, "source_uri": source_uri}
    return docs


def read_names(db):
    names = {}
    for doc_id, node_id, name in db.execute("select doc_id, node_id, name from node"):
        names[(doc_id, node_id)] = name
    return names


def fact_lines(db):
    """(doc_id, fact_id, the fact's line, its quote) for every fact."""
    docs = read_titles(db)
    names = read_names(db)
    fields = ["doc_id", "fact_id", "subject", "subject_name", "predicate", "object", "object_is_node", "qualifiers",
              "occurred_at", "direction", "quote"]
    lines = []
    for row in db.execute(f"select {', '.join(fields)} from fact order by doc_id, fact_id"):
        f = dict(zip(fields, row))
        local = {f["subject"]: names.get((f["doc_id"], f["subject"]), f["subject"]),
                 f["object"]: names.get((f["doc_id"], f["object"]), f["object"])}
        lines.append((f["doc_id"], f["fact_id"], fact_line(f, local, docs[f["doc_id"]]), f["quote"]))
    return lines


def document_texts(db):
    """(record, doc_id, record_id, text) for every fact, cell and abstract row, in order."""
    docs = read_titles(db)
    names = read_names(db)
    rows = []
    for doc_id, fact_id, line, quote in fact_lines(db):
        rows.append(("fact", doc_id, fact_id, line))
    for doc_id, node_id, unit_id, text in db.execute("select doc_id, node_id, unit_id, text from cell order by rowid"):
        for s in sentences(text):
            rows.append(("cell", doc_id, f"{node_id}@{unit_id}", prefix(names.get((doc_id, node_id), node_id), docs[doc_id]) + s))
    for doc_id, node_id, text in db.execute("select doc_id, node_id, text from abstract order by rowid"):
        for s in sentences(text):
            rows.append(("abstract", doc_id, node_id, prefix(names.get((doc_id, node_id), node_id), docs[doc_id]) + s))
    return rows


def without_tags(summary):
    """A parent's summary with the [node id] taken off the front of each line."""
    lines = []
    for line in summary.split("\n"):
        if line.startswith("[") and "] " in line:
            line = line.split("] ", 1)[1]
        lines.append(line)
    return "\n".join(lines)


def parent_texts(db):
    """(record, doc_id, record_id, text) for every sentence of a parent of two or more."""
    rows = []
    query = """select parent_id, name, summary from parent where parent_id in
                 (select parent_id from instance_of group by parent_id having count(*) > 1)
                 and name != 'user' order by parent_id"""
    for parent_id, name, summary in db.execute(query):
        for s in sentences(without_tags(summary)):
            rows.append(("parent", None, str(parent_id), f"{name}: " + plain(s)))
    return rows

## Block 7: the embedding sidecar

Each dump embeds its own rows and saves them. After the last dump the rows are stacked, with the
parents' summary sentences added, into one float16 `.npy` beside the store, and the row map and the
header are written.

```
embedded(path, texts):
    if the vectors are on disk, load them
    otherwise embed the texts and save them

finish_sidecar:
    stack every dump's rows, then the parents' summary sentences
    embed again the rows 0.5 words differently, and write their search lines
    write the array, the row map and the header
```

Each dump's vectors are kept on disk, so a session that attaches a 0.4 run replays its decisions with
no calls and embeds only what 0.5 changed: the rows named by `reworded`, and the parents' sentences,
which are always embedded here.

In [ ]:
EMBEDDER = None


def embed(texts):
    """Unit vectors for texts, float32. The model loads on first use."""
    global EMBEDDER
    if EMBEDDER is None:
        from fastembed import TextEmbedding
        EMBEDDER = TextEmbedding(MODEL)
    return numpy.asarray(list(EMBEDDER.embed(texts, batch_size=256)), dtype=numpy.float32)


def embedded(path, texts):
    """The vectors of texts, kept at `path` so a stopped run does not embed them again."""
    if path.exists():
        return numpy.load(path)
    vectors = embed(texts)
    numpy.save(path, vectors)
    return vectors


def sidecar_rows(db, dumps):
    """The rows of each dump, in the order they will sit in the array."""
    dump_of = {}
    rows = []
    for number, dump in enumerate(dumps):
        rows.append([])
        for doc_id in dump:
            dump_of[doc_id] = number
    for row in document_texts(db):
        rows[dump_of[row[1]]].append(row)
    return rows


def texts_only(rows):
    texts = []
    for row in rows:
        texts.append(row[3])
    return texts


def reworded(db):
    """The rows 0.4 wrote that 0.5 words differently: a fact read backwards, or a chat's marked text."""
    marked = "select doc_id from document where title like 'answer#_%' escape '#' or title like '%#_abs%' escape '#'"
    facts = f"""select doc_id, fact_id from fact where direction = 'inverse'
                  or (quote is null and doc_id in ({marked}))
                  or (predicate = 'belongs_to_history' and object like '%#_abs' escape '#')"""
    keys = set()
    for doc_id, fact_id in db.execute(facts):
        keys.add(("fact", doc_id, fact_id))
    for doc_id, node_id in db.execute(f"select doc_id, node_id from abstract where doc_id in ({marked})"):
        keys.add(("abstract", doc_id, node_id))
    return keys


def rewrite_rows(db, vectors, every_row):
    """Those rows embedded again in the array, and their facts' lines in the search table."""
    keys = reworded(db)
    places = []
    texts = []
    for i, row in enumerate(every_row):
        if (row[0], row[1], row[2]) in keys:
            places.append(i)
            texts.append(row[3])
    vectors[places] = embed(texts)

    lines = {}
    for doc_id, fact_id, line, quote in fact_lines(db):
        if ("fact", doc_id, fact_id) in keys:
            if quote:
                line += "\n" + quote
            lines[("fact", doc_id, fact_id)] = line
    updates = []
    for rowid, doc_id, record_id in db.execute("select rowid, doc_id, record_id from search where record = 'fact'"):
        if ("fact", doc_id, record_id) in lines:
            updates.append((lines[("fact", doc_id, record_id)], rowid))
    db.executemany("update search set text = ? where rowid = ?", updates)
    print(f"reworded: {len(texts)} rows embedded again, {len(updates)} search lines")


def finish_sidecar(store, rows):
    """Every dump's vectors and the parents' sentences as one array, with its map and header."""
    db = sqlite3.connect(store)
    chunks = []
    for number in range(1, DUMPS + 1):
        chunks.append(numpy.load(OUT / "vectors" / f"rows-{number:02d}.npy"))
    added = parent_texts(db)
    chunks.append(embed(texts_only(added)))
    vectors = numpy.vstack(chunks)
    every_row = []
    for dump_rows in rows:
        every_row += dump_rows
    every_row += added
    rewrite_rows(db, vectors, every_row)
    numpy.save(store.with_suffix(".npy"), vectors.astype(numpy.float16))
    row_map = []
    for i, row in enumerate(every_row):
        row_map.append((i, row[0], row[1], row[2]))
    db.execute("delete from vec_row")
    db.execute("delete from vec_header")
    db.executemany("insert into vec_row values (?,?,?,?)", row_map)
    db.execute("insert into vec_header values (?,?)", (MODEL, DIMENSION))
    db.commit()
    db.close()
    print(f"sidecar: {len(every_row)} rows")

## Block 8: reading the corpus

An entity to place is a node that is not a chat's user. A document's own node counts when the
document has a real title. Each is read once with its name, aliases, kind, summary or facts, the
entities it shares units with (its cast), and its `is_a` links inside the document. A cast name's
rarity weighs the second co-occurrence score: a name in every document weighs almost nothing.

A chat's speaker is the person it belongs to, the folder its sessions sit in. The judge is told only
whether two conversations share a speaker, never who.
Each speaker's `user` nodes are one parent, known from the record with no judge and no call.

```
for each node (not a chat's user, not an untitled document's own node):
    keep its name, kind, aliases, abstract and facts
    note its is_a links to other entities of the document
for each chat's user: note it under its speaker
for each unit: each entity's cast gains the other entities in that unit
rarity(name) = log((documents + 1) / (documents holding the name + 1))
```

In [ ]:
def read_children(db, docs):
    """Every entity to place, with its aliases, abstract, facts and is_a links."""
    children = {}
    names = {}
    for doc_id, node_id, name, kind in db.execute("select doc_id, node_id, name, kind from node"):
        document = node_id.endswith(":doc")
        names[(doc_id, node_id)] = name
        if fold(name) == "user" and docs[doc_id]["chat"]:
            continue
        if document and is_identifier(docs[doc_id]["title"], docs[doc_id]["source_uri"]):
            continue
        children[(doc_id, node_id)] = {"doc_id": doc_id, "node_id": node_id, "name": name, "kind": kind, "aliases": [],
                                       "facts": [], "abstract": None, "cast": set(), "links": set(), "document": document}
    for doc_id, alias, node_id in db.execute("select doc_id, alias, node_id from alias"):
        child = children.get((doc_id, node_id))
        if child and alias != child["name"]:
            child["aliases"].append(alias)
    for doc_id, node_id, text in db.execute("select doc_id, node_id, text from abstract"):
        if (doc_id, node_id) in children:
            children[(doc_id, node_id)]["abstract"] = text
    query = "select doc_id, subject, predicate, object, object_is_node, qualifiers, quote from fact order by doc_id, fact_id"
    for doc_id, subject, predicate, obj, is_node, qualifiers, quote in db.execute(query):
        child = children.get((doc_id, subject))
        if child is None:
            continue
        if quote is None and not child["document"]:
            continue
        target = None
        value = obj
        if is_node:
            target = children.get((doc_id, obj))
            value = names.get((doc_id, obj), obj)
        line = f"{predicate.replace('_', ' ')} {value}"
        if qualifiers:
            line += f" ({qualifiers})"
        child["facts"].append(line)
        if target is not None and predicate == "is_a":
            child["links"].add(obj)
            target["links"].add(subject)
    return children


def read_casts(step1, children):
    """Each entity's cast: the folded names of the entities it shares a unit with."""
    members = {}
    for e in read_jsonl(step1 / "edges.jsonl"):
        if e["predicate"] == "appears_in" and (e["doc_id"], e["subject"]) in children:
            for unit in e.get("units") or []:
                members.setdefault((e["doc_id"], unit), []).append(e["subject"])
    for (doc_id, unit), ids in members.items():
        names = set()
        for i in ids:
            names.add(fold(children[(doc_id, i)]["name"]))
        for i in ids:
            children[(doc_id, i)]["cast"] |= names - {fold(children[(doc_id, i)]["name"])}
        whole = children.get((doc_id, f"{doc_id[:8]}:doc"))
        if whole is not None:
            whole["cast"] |= names


def read_speakers(db, docs):
    """speaker -> the keys of the `user` node of each of their conversations, oldest first."""
    speakers = {}
    query = """select n.doc_id, n.node_id from node n join document d on d.doc_id = n.doc_id
                 where lower(n.name) = 'user' order by d.occurred_at, n.doc_id"""
    for doc_id, node_id in db.execute(query):
        if docs[doc_id]["speaker"] is not None:
            speakers.setdefault(docs[doc_id]["speaker"], []).append((doc_id, node_id))
    return speakers


def read_corpus(db, step1):
    """The documents, the entities and each name's rarity, as one dict the later blocks share."""
    chats = set()
    for (doc_id,) in db.execute(CHATS):
        chats.add(doc_id)
    docs = {}
    for doc_id, title, source_uri, occurred_at in db.execute("select doc_id, title, source_uri, occurred_at from document"):
        speaker = None
        if doc_id in chats:
            speaker = PurePosixPath(source_uri).parent.name
        docs[doc_id] = {"title": title, "source_uri": source_uri, "occurred_at": occurred_at, "chat": doc_id in chats,
                        "speaker": speaker}
    children = read_children(db, docs)
    read_casts(step1, children)
    holders = {}
    for child in children.values():
        holders.setdefault(fold(child["name"]), set()).add(child["doc_id"])
    rarity = {}
    for name, doc_ids in holders.items():
        rarity[name] = math.log((len(docs) + 1) / (len(doc_ids) + 1))
    return {"docs": docs, "children": children, "rarity": rarity, "speakers": read_speakers(db, docs)}


def child_text(child, doc):
    """What the judge, the parent writer and the vector see of an entity."""
    lines = [f"name: {child['name']}", f"kind: {child['kind']}"]
    if child["aliases"]:
        lines.append("also called: " + "; ".join(child["aliases"][:15]))
    if not is_identifier(doc["title"], doc["source_uri"]):
        lines.append(f"document: {doc['title']}")
    if child["abstract"]:
        lines.append("summary: " + child["abstract"])
    elif child["facts"]:
        lines.append("facts: " + "; ".join(child["facts"][:FACTS_SHOWN]))
    return "\n".join(lines)


def most_facts_first(keys, children):
    ranked = []
    for key in keys:
        ranked.append((-len(children[key]["facts"]), key))
    ranked.sort()
    ordered = []
    for _, key in ranked:
        ordered.append(key)
    return ordered


def best_first(scored):
    """The items of (score, item) pairs, highest score first; a tie keeps its order."""
    rows = []
    for position, (score, item) in enumerate(scored):
        rows.append((-score, position, item))
    rows.sort()
    items = []
    for _, _, item in rows:
        items.append(item)
    return items

## Block 9: the two prompts

The prompts state rules. The judge sees texts and casts as names, never scores. The writer may only
pick a name an instance carries, and the code refuses any other. Across two speakers' conversations,
an entity known only through its speaker is different unless a name or a fact that does not depend on
the speaker identifies one public thing.

```
JUDGE: are these two entities one thing? Say different when nothing connects them.
       Told whether two conversations share a speaker; what is known only through a speaker
       belongs to that speaker's life.
WRITE: name this parent from its instances, and write one line per instance.
```

In [ ]:
JUDGE = """You are deciding whether two entities are the same person, place, thing or concept, or different
things with a similar name or description. Each is described only by what its document says. Under an
entity, "also united with it" lists other documents' instances already judged to be the same thing as
it; they are context for what the entity is, and the question is not about them. Say same only when
the first entity and the second entity themselves are one thing, even under different names, spellings
or roles, and even when one document knows it under a name the other never uses. Say different when
they are different things that share a name or a word, when one is a part, a possession, a place or an
event of the other, or when nothing in their own descriptions connects them. A work is different from
the person, place or thing it is named for or tells about. An entity that is a whole document is a
work, whatever its title names, and it is one thing with an entity in another document that speaks of
that same work. A shared role, trait or activity does not by itself connect two entities. A
disagreement in kind or role is not by itself a reason to say different; a contradiction in identity is. The reason must speak of the first entity and the second entity, not of
the instances united with them.

A conversation is one person's own account. When the two entities come from different people's
conversations, an entity known only through its relation to its speaker belongs to that speaker's life:
say different unless a name or a fact that does not depend on the speaker identifies one public thing.
A given name alone identifies no one. The name of a known company, product, platform, place or published
work does identify one public thing, and needs no further fact.

The first entity:
{first}
Appears beside: {first_cast}

The second entity:
{second}
Appears beside: {second_cast}

Where they come from: {source}
What they are: {work}

Reply with a JSON object: {{"same": true or false, "reason": "<one sentence>"}}."""

WRITE = """A global entity is described by its instances, the entities that documents hold of it. Below are the
instances of one global entity, numbered. Give it a name and a kind, and write one line per instance
saying what part that instance played in its document. The name must be one the instances actually use
(a name or an alias listed below), the one a reader would look for first. The kind is one or two words.
Each line says only what that instance's description says, in the third person, naming the document.

{instances}

Reply with a JSON object: {{"name": "<name>", "kind": "<kind>", "lines": [<{n} strings, one per instance, in order>]}}."""

## Block 10: the order and the signals

A document set is the source folder a document came from, which any corpus has. Each set is sorted
by date and cut into DUMPS equal runs. A dump takes one run from every set and is itself sorted by
date. So each dump is a mix of everything, and inside any one set time only moves forward.

A shared name counts only when it is written as a name: a capital or a digit anywhere in it, not
counting a leading article. An entity with no such name can still be offered at cosine .80. Each pair put to the judge is scored on its signals and logged.

```
read_dumps:
    for each source folder:
        sort its documents by (year, date as written, path)
        document at position p of n goes to dump floor(p * 10 / n)
    sort each dump the same way

names of an entity = its name and aliases that have a capital or a digit
```

In [ ]:
def year_of(date):
    """The year a date opens with, which may be negative. An undated document sorts last."""
    date = date or ""
    sign = 1
    if date.startswith("-"):
        sign = -1
        date = date[1:]
    digits = ""
    for c in date:
        if not c.isdigit():
            break
        digits += c
    if not digits:
        return 10 ** 6
    return sign * int(digits)


def by_date(doc_ids, docs):
    """Documents sorted by year, date as written, path."""
    rows = []
    for d in doc_ids:
        rows.append((year_of(docs[d]["occurred_at"]), docs[d]["occurred_at"] or "", docs[d]["source_uri"], d))
    rows.sort()
    ordered = []
    for row in rows:
        ordered.append(row[3])
    return ordered


def source_set(doc):
    """The folder a document came from."""
    return PurePosixPath(doc["source_uri"]).parent.as_posix()


def read_dumps(docs):
    """The corpus as DUMPS lists of document ids, each a date-ordered run of every source folder."""
    sets = {}
    for doc_id, doc in docs.items():
        sets.setdefault(source_set(doc), []).append(doc_id)
    dumps = []
    for number in range(DUMPS):
        dumps.append([])
    for members in sets.values():
        for position, doc_id in enumerate(by_date(members, docs)):
            dumps[position * DUMPS // len(members)].append(doc_id)
    ordered = []
    for dump in dumps:
        ordered.append(by_date(dump, docs))
    return ordered


def proper(alias):
    """Written as a name: a capital or a digit anywhere, not counting a leading article."""
    words = alias.split()
    if words and words[0].casefold() in ("the", "a", "an"):
        words = words[1:]
    for word in words:
        for c in word:
            if c.isupper() or c.isdigit():
                return True
    return False


def lexical_names(child):
    """The forms a name match may fire on. A name in Greek script folds to nothing and matches nothing."""
    names = set()
    for alias in [child["name"]] + child["aliases"]:
        if proper(alias):
            names.add(fold(alias))
    names.discard("")
    return names


def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def rare_jaccard(a, b, rarity):
    """Jaccard with each name weighted by its rarity over the loaded documents."""
    either = 0.0
    for name in a | b:
        either += rarity.get(name, 0.0)
    if not either:
        return 0.0
    both = 0.0
    for name in a & b:
        both += rarity.get(name, 0.0)
    return both / either


def score_pair(a, b, sim, rarity, identity=0):
    """One pair on every signal; `identity` marks an is_a link inside a document."""
    lexical = 0.0
    if lexical_names(a) & lexical_names(b):
        lexical = 1.0
    return {"a": (a["doc_id"], a["node_id"]), "b": (b["doc_id"], b["node_id"]), "lexical": lexical, "vector": sim,
            "cast": jaccard(a["cast"], b["cast"]), "cast_rare": rare_jaccard(a["cast"], b["cast"], rarity),
            "identity": identity, "verdict": None, "reason": None}

## Block 11: the judge

One call about a pair. The first side is the entity being placed. The second is the placed entity
the search found, with up to five others of its parent underneath as context. The judge is the gate:
nothing joins on a score. A pair with a chat on either side goes to Luna, the rest to Terra, whatever
the parent's other instances are. A call that fails after its retries counts as no verdict, and the entity goes on to its next parent.

```
judge(pair):
    show both entities, the other instances of the parent, and each one's cast
    ask: same or different, with a reason
    record the verdict; no reply counts as no verdict
```

In [ ]:
def side_text(key, members, corpus):
    """One side as the judge sees it."""
    children = corpus["children"]
    docs = corpus["docs"]
    text = child_text(children[key], docs[key[0]])
    others = []
    for member in members:
        if member != key:
            others.append(member)
    shown = most_facts_first(others, children)[:SIDE_SHOWN - 1]
    if shown:
        lines = []
        for m in shown:
            about = children[m]["abstract"] or "; ".join(children[m]["facts"][:4]) or ""
            lines.append(f"  - {children[m]['name']} ({children[m]['kind']}) in {document_name(docs[m[0]])}: " + about[:300])
        text += "\nalso united with it, in other documents (context only):\n" + "\n".join(lines)
    if len(members) - 1 > len(shown):
        text += f"\n  - and {len(members) - 1 - len(shown)} more"
    return text


def cast_line(names, other):
    """The names an entity appears beside, those it shares with the other side first."""
    if not SHOW_CAST:
        return "(not shown)"
    shown = sorted(names & other) + sorted(names - other)
    return ", ".join(shown[:25])


def source_line(a, b, docs):
    """Whether two entities come from one speaker, from two, or not both from conversations."""
    first = docs[a[0]]["speaker"]
    second = docs[b[0]]["speaker"]
    if first is None or second is None:
        return "they do not both come from conversations."
    if first == second:
        return "both come from one person's conversations."
    return "they come from two different people's conversations."


def work_line(a, b, children):
    """Which of the two entities is a whole document."""
    if children[a]["document"] and children[b]["document"]:
        return "both entities are whole documents."
    if children[a]["document"]:
        return "the first entity is a whole document."
    if children[b]["document"]:
        return "the second entity is a whole document."
    return "neither entity is a whole document."


def judge_pair(pair, side_a, side_b, corpus):
    children = corpus["children"]
    cast_a = children[pair["a"]]["cast"]
    cast_b = children[pair["b"]]["cast"]
    prompt = JUDGE.format(first=side_text(pair["a"], side_a, corpus), first_cast=cast_line(cast_a, cast_b),
                          second=side_text(pair["b"], side_b, corpus), second_cast=cast_line(cast_b, cast_a),
                          source=source_line(pair["a"], pair["b"], corpus["docs"]),
                          work=work_line(pair["a"], pair["b"], children))
    model = TERRA
    if corpus["docs"][pair["a"][0]]["chat"] or corpus["docs"][pair["b"][0]]["chat"]:
        model = LUNA
    reply = generate(prompt, {"same": bool, "reason": str}, "judge", model=model, effort="medium")
    if reply is None:
        pair["verdict"] = "no reply"
    elif reply["same"]:
        pair["verdict"] = "same"
        pair["reason"] = reply["reason"]
    else:
        pair["verdict"] = "different"
        pair["reason"] = reply["reason"]
    return pair

## Block 12: placing a document

The graph is a dict that says which entities are placed and under which parent. A parent is named
here by the entity that founded it. Entities of one document are not offered to each other in the
rounds, and nothing placed in a document is visible until it is
finished. Up to WORKERS documents run side by side, each seeing the graph as it stood when it started.
Each document's decisions are logged as one line, so a stopped run replays the log with no calls.

```
offer(entity):
    parents holding a placed entity with a shared name, closest first
    then parents holding a placed entity at cosine .80 or more, closest first
    keep the best 5

place(document):
    round 1 to 5, all unplaced entities at once:
        judge each against its next offered parent; "same" attaches it
    judge each is_a link inside the document that the rounds left apart; "same" unites their parents
    an entity still unplaced founds a parent
    log the document

join(dump):
    for each parent that changed in this dump:
        other parents sharing a name with it, closest first, not yet compared, at most 5
    judge those pairs; "same" unites the two parents
```

In [ ]:
GRAPH_LOCK = threading.Lock()


def pair_of(a, b):
    """Two parents as one unordered pair."""
    if a < b:
        return (a, b)
    return (b, a)


def new_graph(keys):
    """An empty graph over every entity. `keys` gives each entity its row in the vector array."""
    index = {}
    for i, key in enumerate(keys):
        index[key] = i
    return {"keys": keys,
            "index": index,
            "vectors": numpy.zeros((len(keys), DIMENSION), dtype=numpy.float32),
            "rows": [],             # the row of every placed entity
            "at": {},               # a row -> its place in "rows"
            "parent": {},           # entity -> the entity that founded its parent
            "members": {},          # founder -> its entities
            "named": {},            # name -> the rows of placed entities carrying it
            "changed": set(),       # founders that gained a member in this dump
            "compared": set()}      # pairs of parents already judged


def place_entity(graph, key, founder, children):
    row = graph["index"][key]
    graph["at"][row] = len(graph["rows"])
    graph["rows"].append(row)
    graph["parent"][key] = founder
    graph["members"].setdefault(founder, []).append(key)
    graph["changed"].add(founder)
    for name in lexical_names(children[key]):
        graph["named"].setdefault(name, []).append(row)


def unite(graph, a, b):
    """Two parents become one, under the parent of the first."""
    first = graph["parent"][a]
    second = graph["parent"][b]
    if first == second:
        return
    for child in graph["members"][second]:
        graph["parent"][child] = first
    graph["members"][first] += graph["members"].pop(second)
    graph["changed"].add(first)
    graph["changed"].discard(second)


def note_offer(best, row, named, sims, graph):
    """Keep the strongest placed entity found for each parent."""
    cosine = float(sims[graph["at"][row]])
    key = graph["keys"][row]
    founder = graph["parent"][key]
    if founder not in best or (named, cosine) > best[founder][:2]:
        best[founder] = (named, cosine, key)


def candidates(key, sims, graph, children):
    """The parents to try for one entity, best first, as (founder, placed entity, cosine)."""
    best = {}
    for name in lexical_names(children[key]):
        for row in graph["named"].get(name, ()):
            note_offer(best, row, True, sims, graph)
    close = numpy.nonzero(sims >= VECTOR_FLOOR)[0]
    for place in close[numpy.argsort(-sims[close])]:
        if len(best) >= TRIES:
            break
        note_offer(best, graph["rows"][place], False, sims, graph)
    ranked = []
    for founder, (named, cosine, found) in best.items():
        ranked.append((named, cosine, founder, found))
    ranked.sort(reverse=True)
    offers = []
    for named, cosine, founder, found in ranked[:TRIES]:
        offers.append((founder, found, cosine))
    return offers


def write_line(line):
    with LOG_LOCK:
        with DECISIONS.open("a", encoding="utf-8") as f:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")


def judge_rounds(group, offers, graph, corpus):
    """Each unplaced entity against its next parent, up to TRIES rounds. (target, pairs, ruled apart)."""
    children = corpus["children"]
    target = {}
    for key in group:
        target[key] = None
    pairs = []
    apart = []
    for attempt in range(TRIES):
        jobs = []
        chosen = []
        for key in group:
            if target[key] is None and attempt < len(offers[key]):
                founder, found, cosine = offers[key][attempt]
                pair = score_pair(children[key], children[found], cosine, corpus["rarity"])
                side_b = [found]
                for member in graph["members"][founder]:
                    if member != found:
                        side_b.append(member)
                jobs.append((pair, [key], side_b, corpus))
                chosen.append((key, founder))
        if not jobs:
            break
        for (key, founder), pair in zip(chosen, in_parallel(judge_pair, jobs)):
            pairs.append(pair)
            if pair["verdict"] == "same":
                target[key] = founder
            elif pair["verdict"] == "different":
                apart.append((key, founder))
    return target, pairs, apart


def judge_links(doc_id, group, target, graph, corpus):
    """An is_a link inside the document, when the rounds left it apart. (pairs, sibling, parents to unite)."""
    children = corpus["children"]
    jobs = []
    links = []
    for key in group:
        for node_id in children[key]["links"]:
            other = (doc_id, node_id)
            if key < other and other in target and (target[key] is None or target[key] != target[other]):
                cosine = float(graph["vectors"][graph["index"][key]] @ graph["vectors"][graph["index"][other]])
                pair = score_pair(children[key], children[other], cosine, corpus["rarity"], identity=1)
                jobs.append((pair, [key], [other], corpus))
                links.append((key, other))
    pairs = []
    sibling = {}
    united = []
    for (key, other), pair in zip(links, in_parallel(judge_pair, jobs)):
        pairs.append(pair)
        if pair["verdict"] == "same":
            if target[key] is None and target[other] is not None:
                target[key] = target[other]
            elif target[other] is None and target[key] is not None:
                target[other] = target[key]
            elif target[key] is None and target[other] is None and other not in sibling:
                sibling[other] = key
            elif target[key] is not None and target[other] is not None:
                united.append((target[key], target[other]))
    return pairs, sibling, united


def place_document(doc_id, group, graph, corpus):
    """Every entity of one document placed in the graph, and the document's line written."""
    rows = []
    for key in group:
        rows.append(graph["index"][key])
    offers = {}
    with GRAPH_LOCK:
        sims = graph["vectors"][rows] @ graph["vectors"][graph["rows"]].T
        for i, key in enumerate(group):
            offers[key] = candidates(key, sims[i], graph, corpus["children"])
    target, pairs, apart = judge_rounds(group, offers, graph, corpus)
    link_pairs, sibling, united = judge_links(doc_id, group, target, graph, corpus)
    pairs += link_pairs
    placed = []
    for key in group:
        founder = target[key]
        if founder is None:                      # founds a parent, or joins the sibling that does
            founder = key
            while founder in sibling:
                founder = sibling[founder]
        placed.append((key, founder))
    ruled_apart = []
    for key, founder in apart:
        if target[key] is None:
            ruled_apart.append([key[1], founder[0], founder[1]])
    logged = []
    for key, founder in placed:
        logged.append([key[1], founder[0], founder[1]])
    with GRAPH_LOCK:
        for node_id, founder_doc, founder_node in ruled_apart:  # a parent the judge kept apart is not asked again at the join
            graph["compared"].add(pair_of((doc_id, node_id), (founder_doc, founder_node)))
        for key, founder in placed:
            place_entity(graph, key, founder, corpus["children"])
        for a, b in united:
            unite(graph, a, b)
        write_line({"doc": doc_id, "placed": logged, "ruled_apart": ruled_apart, "united": united, "pairs": pairs})


def place_documents(todo, groups, graph, corpus):
    """A dump's documents, started in date order, WORKERS side by side."""
    pool = ThreadPoolExecutor(max_workers=WORKERS)
    futures = []
    for doc_id in todo:
        futures.append(pool.submit(place_document, doc_id, groups[doc_id], graph, corpus))
    try:
        for future in as_completed(futures):
            future.result()
            check_time()
    except StopRun:                              # documents under way finish, the rest wait for the next session
        pool.shutdown(wait=True, cancel_futures=True)
        raise
    pool.shutdown()


def shared_name_pairs(graph, children):
    """For each changed parent, up to TRIES others sharing a name, closest first, not yet compared."""
    found = set()
    for founder in sorted(graph["changed"]):
        names = set()
        for member in graph["members"][founder]:
            names |= lexical_names(children[member])
        others = set()
        for name in names:
            for row in graph["named"].get(name, ()):
                others.add(graph["parent"][graph["keys"][row]])
        others.discard(founder)
        vector = graph["vectors"][graph["index"][founder]]
        ranked = []
        for other in others:
            ranked.append((float(vector @ graph["vectors"][graph["index"][other]]), other))
        ranked.sort(reverse=True)
        fresh = []
        for cosine, other in ranked:
            if pair_of(founder, other) not in graph["compared"]:
                fresh.append((cosine, other))
        for cosine, other in fresh[:TRIES]:
            a, b = pair_of(founder, other)
            found.add((cosine, a, b))
    return sorted(found, reverse=True)


def join_parents(dump, graph, corpus):
    """One pass over the parents that changed. A union is applied strongest first and logged."""
    children = corpus["children"]
    found = shared_name_pairs(graph, children)
    jobs = []
    for cosine, a, b in found:
        pair = score_pair(children[a], children[b], cosine, corpus["rarity"])
        jobs.append((pair, graph["members"][a], graph["members"][b], corpus))
    pairs = in_parallel(judge_pair, jobs)
    joined = []
    settled = []
    for pair, (_, a, b) in zip(pairs, found):
        if pair["verdict"] != "no reply":
            settled.append([a, b])
            graph["compared"].add(pair_of(a, b))
        if pair["verdict"] == "same":
            joined.append([a, b])
            unite(graph, a, b)
    write_line({"joined_dump": dump, "joined": joined, "compared": settled, "pairs": pairs})
    print(f"  join: {len(found)} pairs of parents judged, {len(joined)} united")


def replay(graph, children):
    """What earlier sessions decided, read back with no calls."""
    done = {"documents": set(), "joined": set(), "dumps": set()}
    if not DECISIONS.exists():
        return done
    for row in read_jsonl(DECISIONS):
        if "doc" in row:
            for node_id, founder_doc, founder_node in row["ruled_apart"]:
                graph["compared"].add(pair_of((row["doc"], node_id), (founder_doc, founder_node)))
            for node_id, founder_doc, founder_node in row["placed"]:
                place_entity(graph, (row["doc"], node_id), (founder_doc, founder_node), children)
            for a, b in row.get("united", []):
                unite(graph, tuple(a), tuple(b))
            done["documents"].add(row["doc"])
        elif "joined_dump" in row:
            for a, b in row["compared"]:
                graph["compared"].add(pair_of(tuple(a), tuple(b)))
            for a, b in row["joined"]:
                unite(graph, tuple(a), tuple(b))
            done["joined"].add(row["joined_dump"])
        elif "dump_done" in row:
            graph["changed"].clear()
            done["dumps"].add(row["dump_done"])
    return done

## Block 13: writing the parents

A parent stands over the entities placed under it. One instance is copied with no call. Two or more
get one call that picks the name and kind from what the instances carry and writes one line per
instance. A chat's lines name the conversation by its date, never its session id. A speaker's parent
takes no call: its lines come from the record, and they are not embedded.

Parents are rewritten once per dump, and only those whose members changed. The judge never reads a
written parent. Each step's writes are
logged in `parents.jsonl`, so a session that stops between steps does not pay for them twice. The
tables are written after each dump.

```
rewrite(dump):
    for each parent whose members differ from its last written version:
        one instance  -> copy it
        two or more   -> one call: name, kind, a line per instance

tables: replace parent and instance_of
```

In [ ]:
def alone(members, children, docs):
    """A parent of one instance, copied from it."""
    first = children[members[0]]
    title = document_name(docs[first["doc_id"]])
    line = f"In {title}: " + (first["abstract"] or "; ".join(first["facts"][:FACTS_SHOWN]) or first["name"])
    return {"name": first["name"], "kind": first["kind"], "aliases": {first["name"], *first["aliases"]},
            "summary": [(members[0], line)], "children": members}


def instance_lines(members, children, docs, reply):
    """One summary line per instance: the call's line when it wrote one, else a plain line."""
    lines = []
    if reply:
        lines = reply["lines"]
    summary = []
    for n, key in enumerate(members):
        line = None
        if n < len(lines) and isinstance(lines[n], str) and lines[n].strip():
            line = lines[n]
        summary.append((key, line or f"In {document_name(docs[key[0]])}: {children[key]['name']}."))
    return summary


def write_parent(members, corpus):
    """A parent from its instances, most facts first. The call may only pick a name they carry."""
    children = corpus["children"]
    docs = corpus["docs"]
    members = most_facts_first(members, children)
    if len(members) == 1:
        return alone(members, children, docs)
    aliases = set()
    folded = set()
    for key in members:
        for alias in [children[key]["name"]] + children[key]["aliases"]:
            aliases.add(alias)
            folded.add(fold(alias))
    entries = []
    shown = members[:GROUP_SHOWN]
    for n, key in enumerate(shown):
        entries.append(f"[{n + 1}] {child_text(children[key], docs[key[0]])}\ndocument: {document_name(docs[key[0]])}")
    prompt = WRITE.format(instances="\n\n".join(entries), n=len(shown))
    reply = generate(prompt, {"name": str, "kind": str, "lines": list}, "write")
    name = children[members[0]]["name"]
    kind = children[members[0]]["kind"]
    if reply and fold(reply["name"]) in folded:
        name = reply["name"]
    if reply and reply["kind"]:
        kind = reply["kind"]
    summary = instance_lines(members, children, docs, reply)
    return {"name": name, "kind": kind, "aliases": aliases, "summary": summary, "children": members}


def speaker_parents(corpus):
    """One parent per speaker over the `user` of each of their conversations, from the record, with no call."""
    parents = []
    for speaker, keys in sorted(corpus["speakers"].items()):
        summary = []
        for key in keys:
            summary.append((key, f"The user in {document_name(corpus['docs'][key[0]])}."))
        parents.append({"name": "user", "kind": "person", "aliases": {"user"}, "summary": summary, "children": keys})
    return parents


def read_written():
    """The parents written by earlier sessions; a later line replaces an earlier one."""
    written = {}
    if not WRITTEN_PARENTS.exists():
        return written
    for row in read_jsonl(WRITTEN_PARENTS):
        summary = []
        for key, text in row["summary"]:
            summary.append((tuple(key), text))
        members = []
        for key in row["children"]:
            members.append(tuple(key))
        written[tuple(row["key"])] = {"name": row["name"], "kind": row["kind"], "aliases": set(row["aliases"]),
                                      "summary": summary, "children": members}
    return written


def rewrite_parents(parents, written, corpus):
    """Each parent whose members changed is written again; one that took a call is logged."""
    stale = []
    jobs = []
    for founder, members in parents.items():
        if founder not in written or set(written[founder]["children"]) != set(members):
            stale.append(founder)
            jobs.append((members, corpus))
    results = in_parallel(write_parent, jobs)
    with WRITTEN_PARENTS.open("a", encoding="utf-8") as f:
        for founder, result in zip(stale, results):
            written[founder] = result
            if len(result["children"]) > 1:
                row = {"key": founder, "name": result["name"], "kind": result["kind"], "aliases": sorted(result["aliases"]),
                       "summary": result["summary"], "children": result["children"]}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"  rewrite: {len(stale)} parents changed")


def write_tables(db, parents):
    """The parents and up-edges into the store."""
    db.execute("delete from parent")
    db.execute("delete from instance_of")
    for i, p in enumerate(parents):
        lines = []
        for (_, node_id), line in p["summary"]:          # a node id carries its document's tag
            lines.append(f"[{node_id}] {line}")
        aliases = column(sorted(p["aliases"]))
        db.execute("insert into parent values (?,?,?,?,?)", (i, p["name"], p["kind"], aliases, "\n".join(lines)))
        edges = []
        for key in p["children"]:
            edges.append((key[0], key[1], i))
        db.executemany("insert into instance_of values (?,?,?)", edges)
    db.commit()


def write_pairs(store):
    """Every judged pair, from the decisions log, into build.sqlite."""
    rows = []
    for line in read_jsonl(DECISIONS):
        for p in line.get("pairs", []):
            rows.append((p["a"][0], p["a"][1], p["b"][0], p["b"][1], p["lexical"], p["vector"], p["cast"], p["cast_rare"],
                         p["identity"], p["verdict"], p["reason"]))
    log = sqlite3.connect(store.with_name("build.sqlite"))
    log.execute("delete from pair")
    log.executemany("insert into pair values (?,?,?,?,?,?,?,?,?,?,?)", rows)
    log.commit()
    log.close()

## Block 14: bodies of work

Books and papers are grouped the way entities are placed into parents, with the works as the
entities and the bodies of work as the parents. It follows TopicGPT (Pham et al. 2024), where an LLM
puts each document in an existing topic or founds a new one, merges near-duplicate topics, and makes
subtopics under each topic. The groups offered come from the graph itself, using the related-works
score over shared entities, with abstract similarity beside it.

The score between two works sums over the parents they share:

```
related(a, b) = sum over shared parents e of  salience(e, a) * salience(e, b) * rarity(e)
salience(e, d) = e's share of d, each entity counting 1 plus its facts
rarity(e) = log(documents / (1 + documents holding e)), never below 0
```

The score is divided by the square root of each work's score with itself. A work with no group stays
alone, and a body exists once it has two works.

After the last dump every group is compared again against its closest. Each reason is logged in
`decisions.jsonl`.

```
group(dump):
    for each new work, in date order:
        rank the placed works by related(), and by abstract cosine
        fuse the two rankings, keep the best 5 groups
        for each: does this work belong to this body? the first yes attaches it
        no yes: the work stays alone
    for each group that changed:
        compare it with up to 5 groups sharing an entity, each pair once; yes unites them
    after the last dump: forget the comparisons and compare every group once more
    for each changed body of 2 or more:
        one call names and describes it
        one call sorts its works into subgroups, choosing how many
```

In [ ]:
BELONGS = """You are deciding whether a work belongs to a body of work in a library. A body of work is a set of
works a reader would shelve together: one series, one author, one literary or historical tradition, or
one field of research. Works set in one invented world, or telling one tradition of stories, are one
body whoever wrote them and in whatever language or form. Its works need not treat the same question;
works of one series or one field belong together even when each is about something different. Say yes
when the work belongs to this body. Say no when the only connection is a common word, a common first name, or a broad period or form
such as being old or being a novel, or when nothing connects them.

The work:
Title: {title}
Abstract: {abstract}

The body of work:
Name: {name}
Description: {description}
Works in it ({count}): {titles}

Names they share, strongest first: {shared}

Reply with a JSON object: {{"belongs": true or false, "reason": "<one sentence>"}}."""

SAME_BODY = """You are deciding whether two groups of works in a library are one body of work. A body of work is a set
of works a reader would shelve together: one series, one author, one literary or historical tradition,
or one field of research. Works set in one invented world, or telling one tradition of stories, are one
body whoever wrote them and in whatever language or form; a difference of author, language, form or
period does not separate them. Say yes when they are one body. Say no when the only connection is a common
word, a common first name, or a broad period or form, or when nothing connects them.

The first group:
Name: {first_name}
Description: {first_description}
Works in it: {first_titles}

The second group:
Name: {second_name}
Description: {second_description}
Works in it: {second_titles}

Names they share, strongest first: {shared}

Reply with a JSON object: {{"same": true or false, "reason": "<one sentence>"}}."""

DESCRIBE = """Below are the works of one body of work in a library. Name it the way a librarian would, and describe
it in one sentence broad enough to cover all of them.

{listing}

Reply with a JSON object: {{"name": "<a short name>", "description": "<one sentence>"}}."""

SPLIT = """Below are the works of one body of work in a library, numbered. Sort them into subgroups a reader would
find useful, by what the works are about. Decide how many subgroups it needs. Each work goes in exactly
one subgroup.

{listing}

Reply with a JSON object: {{"subgroups": [{{"name": "<a short name>", "works": [<numbers>]}}]}}."""


def read_library():
    """The bodies of work so far: every group, with the works placed and the pairs compared."""
    if not LIBRARY.exists():
        return {"groups": [], "placed": [], "compared": set(), "next": 0}
    state = json.loads(LIBRARY.read_text(encoding="utf-8"))
    compared = set()
    for pair in state["compared"]:
        compared.add(tuple(pair))
    state["compared"] = compared
    return state


def save_library(state):
    saved = dict(state)
    saved["compared"] = sorted(state["compared"])
    LIBRARY.write_text(json.dumps(saved, ensure_ascii=False), encoding="utf-8")


def holdings(works, graph, children, documents):
    """salience[work][parent], and each parent's rarity over the documents that have arrived."""
    weight = {}
    for doc_id in works:
        weight[doc_id] = {}
    rarity = {}
    for founder, members in graph["members"].items():
        holders = set()
        for m in members:
            holders.add(m[0])
        rarity[founder] = max(0.0, math.log(documents / (1 + len(holders))))
        for m in members:
            if m[0] in weight:
                weight[m[0]][founder] = weight[m[0]].get(founder, 0) + 1 + len(children[m]["facts"])
    salience = {}
    for doc_id, counts in weight.items():
        total = 0
        for n in counts.values():
            total += n
        salience[doc_id] = {}
        for founder, n in counts.items():
            salience[doc_id][founder] = n / total
    return salience, rarity


def related(a, b, salience, rarity):
    total = 0
    for e, share in salience[a].items():
        if e in salience[b]:
            total += share * salience[b][e] * rarity[e]
    return total


def titles_of(group, titles):
    shown = []
    for doc_id in group["works"][-TITLES_SHOWN:]:
        shown.append(titles[doc_id])
    return "; ".join(shown)


def shared_entities(side_a, side_b, salience, rarity, children):
    """The entities two sets of works share, strongest first."""
    strength = {}
    for a in side_a:
        for b in side_b:
            for e, share in salience[a].items():
                if e in salience[b]:
                    strength[e] = strength.get(e, 0.0) + share * salience[b][e] * rarity[e]
    scored = []
    for e, score in strength.items():
        scored.append((score, e))
    names = []
    for e in best_first(scored)[:NAMES_SHOWN]:
        names.append(children[e]["name"])
    return ", ".join(names) or "(none)"


def offered_groups(doc_id, state, closeness, cosines):
    """The groups to try for a work, best first: placed works by the fusion of the two rankings."""
    placed = state["placed"]
    by_entities = []
    by_abstract = []
    for work in placed:
        if closeness[doc_id][work] > 0:
            by_entities.append((closeness[doc_id][work], work))
        by_abstract.append((cosines[doc_id][work], work))
    fused = {}
    for ranking in (best_first(by_entities), best_first(by_abstract)):
        for rank, work in enumerate(ranking):
            fused[work] = fused.get(work, 0.0) + 1 / (RRF_K + rank + 1)
    group_of = {}
    for group in state["groups"]:
        for work in group["works"]:
            group_of[work] = group["number"]
    scored = []
    for work, score in fused.items():
        scored.append((score, work))
    numbers = []
    for work in best_first(scored):
        if group_of[work] not in numbers:
            numbers.append(group_of[work])
        if len(numbers) == TRIES:
            break
    return numbers


def place_work(doc_id, state, shared):
    """One work into a group, or alone."""
    titles, abstracts = shared["titles"], shared["abstracts"]
    by_number = {}
    for group in state["groups"]:
        by_number[group["number"]] = group
    tried = []
    for number in offered_groups(doc_id, state, shared["closeness"], shared["cosines"]):
        group = by_number[number]
        names = shared_entities([doc_id], group["works"], shared["salience"], shared["rarity"], shared["children"])
        prompt = BELONGS.format(title=titles[doc_id], abstract=abstracts.get(doc_id, ""), name=group["name"],
                                description=group["description"], count=len(group["works"]),
                                titles=titles_of(group, titles), shared=names)
        reply = generate(prompt, {"belongs": bool}, "group", model=TERRA, effort="medium")
        tried.append(number)
        if reply:
            write_line({"work": doc_id, "group": group["name"], "belongs": reply["belongs"], "reason": reply.get("reason")})
        if reply and reply["belongs"]:
            group["works"].append(doc_id)
            group["changed"] = True
            return
    number = state["next"]
    state["next"] += 1
    state["groups"].append({"number": number, "name": titles[doc_id], "description": abstracts.get(doc_id, ""), "works": [doc_id],
                            "changed": True, "subgroups": []})
    for other in tried:                          # already ruled against this work, so not judged again
        state["compared"].add((other, number))


def group_pair(a, b):
    """Two groups' numbers as one unordered pair."""
    return (min(a, b), max(a, b))


def closest_groups(first, state, closeness):
    """The groups sharing an entity with `first` and not yet compared with it, closest first, as far as TRIES."""
    scored = []
    for other in state["groups"]:
        if other is first:
            continue
        best = 0
        for a in first["works"]:
            for b in other["works"]:
                best = max(best, closeness[a][b])
        if best > 0 and group_pair(first["number"], other["number"]) not in state["compared"]:
            scored.append((best, other["number"]))
    return best_first(scored)[:TRIES]


def join_groups(state, shared):
    """Each changed group against up to TRIES groups sharing an entity, each pair once."""
    titles = shared["titles"]
    changed = []
    for group in state["groups"]:
        if group["changed"]:
            changed.append(group)
    for first in changed:
        if first not in state["groups"]:
            continue
        by_number = {}
        for group in state["groups"]:
            by_number[group["number"]] = group
        for number in closest_groups(first, state, shared["closeness"]):
            second = by_number[number]
            if second not in state["groups"]:            # united into another a moment ago
                continue
            state["compared"].add(group_pair(first["number"], number))
            names = shared_entities(first["works"], second["works"], shared["salience"], shared["rarity"], shared["children"])
            prompt = SAME_BODY.format(first_name=first["name"], first_description=first["description"],
                                      first_titles=titles_of(first, titles), second_name=second["name"],
                                      second_description=second["description"], second_titles=titles_of(second, titles),
                                      shared=names)
            reply = generate(prompt, {"same": bool}, "group", model=TERRA, effort="medium")
            if reply:
                write_line({"groups": [first["name"], second["name"]], "same": reply["same"], "reason": reply.get("reason")})
            if reply and reply["same"]:
                first["works"] += second["works"]
                first["changed"] = True
                state["groups"].remove(second)


def listing_of(group, titles, abstracts):
    entries = []
    for n, doc_id in enumerate(group["works"]):
        entries.append(f"[{n + 1}] {titles[doc_id]}\n{abstracts.get(doc_id, '')}")
    return "\n\n".join(entries)


def describe_group(group, titles, abstracts):
    prompt = DESCRIBE.format(listing=listing_of(group, titles, abstracts))
    reply = generate(prompt, {"name": str, "description": str}, "group", model=TERRA)
    if reply:
        group["name"] = reply["name"]
        group["description"] = reply["description"]


def split_group(group, titles, abstracts):
    """The subgroups of one body. A work the reply leaves out goes in Other; a repeat is ignored."""
    reply = generate(SPLIT.format(listing=listing_of(group, titles, abstracts)), {"subgroups": list}, "group", model=TERRA)
    subgroups = []
    seen = set()
    wanted = []
    if reply:
        wanted = reply["subgroups"]
    for sub in wanted:
        members = []
        for number in sub.get("works", []):
            if isinstance(number, int) and 1 <= number <= len(group["works"]) and number not in seen:
                seen.add(number)
                members.append(group["works"][number - 1])
        if members:
            subgroups.append({"name": sub.get("name") or "Untitled", "works": members})
    left = []
    for n, doc_id in enumerate(group["works"]):
        if n + 1 not in seen:
            left.append(doc_id)
    if left:
        subgroups.append({"name": "Other", "works": left})
    group["subgroups"] = subgroups


def group_library(new_works, state, graph, corpus, abstracts, documents, last=False):
    """This dump's books and papers into bodies of work, and the state saved. After the last dump every
    group is compared again, since a pair ruled apart when both were thin has more to show now."""
    arrived = state["placed"] + new_works
    if not arrived:
        return
    titles = {}
    for doc_id in arrived:
        titles[doc_id] = corpus["docs"][doc_id]["title"]
    salience, rarity = holdings(arrived, graph, corpus["children"], documents)
    texts = []
    for doc_id in arrived:
        texts.append(abstracts.get(doc_id) or titles[doc_id])
    vectors = embed(texts)
    similar = vectors @ vectors.T
    links = {}
    for a in arrived:
        links[a] = related(a, a, salience, rarity)
    closeness = {}
    cosines = {}
    for i, a in enumerate(arrived):
        closeness[a] = {}
        cosines[a] = {}
        for j, b in enumerate(arrived):
            closeness[a][b] = 0.0
            joint = math.sqrt(links[a] * links[b])
            if joint:
                closeness[a][b] = related(a, b, salience, rarity) / joint
            cosines[a][b] = float(similar[i, j])
    shared = {"titles": titles, "abstracts": abstracts, "salience": salience, "rarity": rarity, "children": corpus["children"],
              "closeness": closeness, "cosines": cosines}
    for doc_id in new_works:
        place_work(doc_id, state, shared)
        state["placed"].append(doc_id)
    join_groups(state, shared)
    if last:
        state["compared"] = set()
        for group in state["groups"]:
            group["changed"] = True
        join_groups(state, shared)
    bodies = 0
    for group in state["groups"]:
        if group["changed"] and len(group["works"]) > 1:
            describe_group(group, titles, abstracts)
            split_group(group, titles, abstracts)
        group["changed"] = False
        if len(group["works"]) > 1:
            bodies += 1
    save_library(state)
    print(f"  library: {bodies} bodies of work, {len(state['groups']) - bodies} works alone; ${SPENT:.2f}")

## Block 15: reading the corpus in

One dump at a time. Everything decided is logged, so any session can stop between documents and the
next goes on from there, with the graph rebuilt from the log.

```
rebuild the graph from decisions.jsonl
for each dump:
    embed its rows and entities (kept if already on disk)
    if the dump is finished: go on to the next
    for each document not yet placed: place it; stop if the time is up
    join; rewrite the changed parents; write the tables
    group this dump's books and papers
    log the dump as finished, and the graph's size in dumps.jsonl
copy the judged pairs into build.sqlite
stack the sidecar, with the parents' sentences
```

In [ ]:
def note_dump(number, documents, graph):
    """The graph after a dump: its size and what the run has spent, one line in dumps.jsonl."""
    shared = 0
    for members in graph["members"].values():
        if len(members) > 1:
            shared += 1
    line = {"dump": number, "documents": documents, "entities": len(graph["rows"]), "parents": len(graph["members"]),
            "parents_with_two_or_more": shared, "cost": round(SPENT, 4)}
    with (OUT / "dumps.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps(line) + "\n")
    print(f"dump {number} of {DUMPS}: {documents} documents, {len(graph['rows'])} entities placed, "
          f"{len(graph['members'])} parents, {shared} with 2+; ${SPENT:.2f}")


def embed_dump(number, rows, entities, graph, corpus):
    """This dump's rows and entities, from disk when an earlier session embedded them."""
    embedded(OUT / "vectors" / f"rows-{number:02d}.npy", texts_only(rows))
    texts = []
    places = []
    for key in entities:
        texts.append(child_text(corpus["children"][key], corpus["docs"][key[0]]))
        places.append(graph["index"][key])
    graph["vectors"][places] = embedded(OUT / "vectors" / f"entities-{number:02d}.npy", texts)


def read_dump(number, dump, graph, groups, done, written, library, corpus, abstracts, arrived, db):
    """One dump from its first unplaced document to its last parent and body of work."""
    todo = []
    for doc_id in dump:
        if doc_id not in done["documents"] and groups.get(doc_id):
            todo.append(doc_id)
    place_documents(todo, groups, graph, corpus)
    if number not in done["joined"]:
        check_time()
        join_parents(number, graph, corpus)
    check_time()
    parents = {}
    for founder, members in graph["members"].items():
        parents[founder] = list(members)
    rewrite_parents(parents, written, corpus)
    finished = []
    for founder in parents:
        finished.append(written[founder])
    write_tables(db, finished + speaker_parents(corpus))
    new_works = []
    for doc_id in dump:
        if not corpus["docs"][doc_id]["chat"] and doc_id not in library["placed"]:
            new_works.append(doc_id)
    group_library(new_works, library, graph, corpus, abstracts, arrived, last=number == DUMPS)
    write_line({"dump_done": number})
    graph["changed"].clear()
    note_dump(number, len(dump), graph)


def read_abstracts(db):
    """doc_id -> the document's own abstract."""
    abstracts = {}
    for doc_id, text in db.execute("select doc_id, text from abstract where node_id = substr(doc_id, 1, 8) || ':doc'"):
        abstracts[doc_id] = text
    return abstracts


def build_graph(store, step1):
    """The dumps in turn, from the store to the finished parents and sidecar."""
    db = sqlite3.connect(store)
    corpus = read_corpus(db, step1)
    keys = sorted(corpus["children"])
    abstracts = read_abstracts(db)
    print(f"{len(corpus['docs'])} documents, {len(keys)} entities")
    groups = {}
    for key in keys:
        groups.setdefault(key[0], []).append(key)
    dumps = read_dumps(corpus["docs"])
    rows = sidecar_rows(db, dumps)
    graph = new_graph(keys)
    done = replay(graph, corpus["children"])
    written = read_written()
    library = read_library()
    print(f"{len(done['documents'])} documents already placed, {len(done['dumps'])} dumps finished; ${SPENT:.2f}")
    (OUT / "vectors").mkdir(exist_ok=True)
    arrived = 0
    for number, dump in enumerate(dumps, 1):
        arrived += len(dump)
        entities = []
        for doc_id in dump:
            entities += groups.get(doc_id, [])
        embed_dump(number, rows[number - 1], entities, graph, corpus)
        if number not in done["dumps"]:
            read_dump(number, dump, graph, groups, done, written, library, corpus, abstracts, arrived, db)
    db.close()
    check_time()
    write_pairs(store)
    finish_sidecar(store, rows)

## Block 16: run

The store, then the corpus, then a receipt. A session that reaches its time stop ends here with its
output saved, and the later blocks wait for the last dump.

```
build the store once
read the corpus in; on a stop, say how to go on and stop here
write the receipt
```

In [ ]:
FINISHED = False

if __name__ == "__main__":
    if STEP1 is None or EXPORT is None:
        raise SystemExit("attach it494-threadatlas-step1 and it494-threadatlas-step0, or set STEP1 and EXPORT")
    if not STORE.exists():
        build_store(STEP1, EXPORT / "documents.jsonl", STORE)
    try:
        build_graph(STORE, STEP1)
        FINISHED = True
    except StopRun as stop:
        print(f"{stop}. Turn this output into a dataset, attach it and run again to go on.")


def count_of(db, query):
    return db.execute(query).fetchone()[0]


def write_receipt():
    """The run's counts and cost, printed and written to receipt.json."""
    db = sqlite3.connect(STORE)
    log = sqlite3.connect(STORE.with_name("build.sqlite"))
    sizes = "select count(*) as n from instance_of group by parent_id"
    tables = {}
    for table in ("document", "node", "fact", "cell", "abstract"):
        tables[table] = count_of(db, f"select count(*) from {table}")
    if count_of(db, "select count(*) from sqlite_master where name = 'chunk'") > 0:
        tables["chunk"] = count_of(db, "select count(*) from chunk")
    receipt = {"version": VERSION, "show_cast": SHOW_CAST, "store": tables,
               "parents": count_of(db, "select count(*) from parent"),
               "parents_with_two_or_more": count_of(db, f"select count(*) from ({sizes}) where n > 1"),
               "largest_group": count_of(db, f"select max(n) from ({sizes})"),
               "up_edges": count_of(db, "select count(*) from instance_of"),
               "pairs": count_of(log, "select count(*) from pair"),
               "pairs_same": count_of(log, "select count(*) from pair where verdict = 'same'"),
               "pairs_no_reply": count_of(log, "select count(*) from pair where verdict = 'no reply'"),
               "vectors": count_of(db, "select count(*) from vec_row"), "cost": round(SPENT, 4)}
    db.close()
    log.close()
    (OUT / "receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
    print(json.dumps(receipt, indent=2))

In [ ]:
if __name__ == "__main__" and FINISHED:
    write_receipt()

## Block 16b: chunks

Every document's text cut into the chunks plain RAG uses, so a search can find a passage no fact was
read from. A chunk is 256 tokens (o200k_base), each starting 32 tokens back from the last one's end,
the benchmark's own cut, applied the same way to every document. The store keeps a chunk as offsets
into Step 0's text, never the text itself. Each chunk gets a keyword row in `search`, so bm25 ranks
it with the facts, and one vector, appended to the sidecar; both are written from the text without
LongMemEval's marks, as every other searched line is. A fact whose quote lies inside a chunk is
linked to it, which is the walk's edge between them. A run whose store already has its chunks skips
this block; the chunks' vectors are kept on disk as they are made, so a stopped run goes on.

```
for each document:
    tokens = its text, tokenized; offsets = where each token starts in the text
    chunk k = tokens [k * 224, k * 224 + 256), as the characters those tokens cover
chunk table, keyword rows, and the facts each chunk holds
embed the chunks, 20,000 at a time, and append them to the array and to vec_row
```

In [ ]:
import bisect

CHUNK_TOKENS = 256
CHUNK_OVERLAP = 32
CHUNK_BATCH = 20000


def chunk_spans(text, encoder):
    """(start, end) character offsets of each chunk of the text."""
    tokens = encoder.encode(text, disallowed_special=())
    if len(tokens) == 0:
        return []
    decoded, offsets = encoder.decode_with_offsets(tokens)
    if decoded != text:                   # a token split inside a character: cut on the decoded text's offsets
        offsets = None
    spans = []
    start = 0
    while start < len(tokens):
        end = min(start + CHUNK_TOKENS, len(tokens))
        if offsets is None:
            first = len(encoder.decode(tokens[:start]))
            last = len(encoder.decode(tokens[:end]))
        else:
            first = offsets[start]
            last = len(text)
            if end < len(tokens):
                last = offsets[end]
        if text[first:last].strip():
            spans.append((first, last))
        if end == len(tokens):
            break
        start += CHUNK_TOKENS - CHUNK_OVERLAP
    return spans


def add_chunks(store, export_documents):
    """The chunk table, its keyword rows, the facts each chunk holds, and the chunks' vectors. The sidecar
    step rewrites vec_row on every run, so the chunks are added again whenever their rows are missing."""
    db = sqlite3.connect(store)
    if db.execute("select count(*) from vec_row where record = 'chunk'").fetchone()[0] > 0:
        print("the store already has its chunks")
        db.close()
        return
    db.execute("drop table if exists chunk")
    db.execute("drop table if exists fact_in_chunk")
    db.execute("delete from search where record = 'chunk'")
    try:
        import tiktoken
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)
        import tiktoken
    encoder = tiktoken.get_encoding("o200k_base")

    wanted = set()
    for (doc_id,) in db.execute("select doc_id from document"):
        wanted.add(doc_id)
    chunks = []                           # (doc_id, chunk_id, start, end, text)
    with Path(export_documents).open(encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row["doc_id"] not in wanted:
                continue
            number = 1
            for start, end in chunk_spans(row["text"], encoder):
                # searched and embedded without LongMemEval's marks, as every other line is; the offsets keep
                # pointing at the text as it is
                chunks.append((row["doc_id"], str(number), start, end, plain(row["text"][start:end])))
                number += 1
    print(f"chunks: {len(chunks)} from {len(wanted)} documents")

    # the vectors first, kept on disk a batch at a time, so a stopped run does not embed them again
    (OUT / "vectors").mkdir(exist_ok=True)
    parts = []
    for at in range(0, len(chunks), CHUNK_BATCH):
        texts = []
        for chunk in chunks[at:at + CHUNK_BATCH]:
            texts.append(chunk[4])
        parts.append(embedded(OUT / "vectors" / f"chunks-{at // CHUNK_BATCH:03d}.npy", texts))
        check_time()

    db.execute("create table chunk (doc_id text, chunk_id text, start integer, end integer, primary key (doc_id, chunk_id))")
    db.execute("create table fact_in_chunk (doc_id text, fact_id text, chunk_id text)")
    rows = []
    searched = []
    for doc_id, chunk_id, start, end, text in chunks:
        rows.append((doc_id, chunk_id, start, end))
        searched.append(("chunk", doc_id, chunk_id, text))
    db.executemany("insert into chunk values (?,?,?,?)", rows)
    db.executemany("insert into search values (?,?,?,?)", searched)

    # a fact belongs to every chunk that holds its whole quote; a document's chunks start in order, so only
    # the chunks starting at or before the quote are looked at, the nearest first
    starts_of = {}
    spans_of = {}
    for doc_id, chunk_id, start, end, text in chunks:
        starts_of.setdefault(doc_id, []).append(start)
        spans_of.setdefault(doc_id, []).append((start, end, chunk_id))
    links = []
    for doc_id, fact_id, start, end in db.execute(
            "select doc_id, fact_id, quote_start, quote_end from fact where quote is not null"):
        if doc_id not in spans_of:
            continue
        k = bisect.bisect_right(starts_of[doc_id], start) - 1
        while k >= 0 and spans_of[doc_id][k][1] >= end:
            links.append((doc_id, fact_id, spans_of[doc_id][k][2]))
            k -= 1
    db.executemany("insert into fact_in_chunk values (?,?,?)", links)
    db.execute("create index fact_chunk on fact_in_chunk (doc_id, fact_id)")
    db.execute("create index chunk_fact on fact_in_chunk (doc_id, chunk_id)")

    # the chunks' rows go after every row the array already has
    first = db.execute("select count(*) from vec_row").fetchone()[0]
    vec_rows = []
    for i in range(len(chunks)):
        vec_rows.append((first + i, "chunk", chunks[i][0], chunks[i][1]))
    db.executemany("insert into vec_row values (?,?,?,?)", vec_rows)
    old = numpy.load(store.with_suffix(".npy"))
    new = numpy.vstack(parts).astype(numpy.float16)
    numpy.save(store.with_suffix(".npy"), numpy.vstack([old, new]))
    db.commit()
    db.close()
    print(f"chunks: {len(chunks)} stored as offsets, {len(links)} fact links, {len(chunks)} vectors appended")

In [ ]:
if __name__ == "__main__" and FINISHED:
    try:
        add_chunks(STORE, EXPORT / "documents.jsonl")
        write_receipt()
    except StopRun as stop:
        FINISHED = False
        print(f"{stop}. Turn this output into a dataset, attach it and run again to go on.")

## Block 17: the Oz key

The external key: Wikipedia's list of the characters L. Frank Baum created (CC BY-SA 4.0,
`jhffmn/it494-threadatlas-oz-key`, built by `scripts/build_oz_key.py`, with 53 aliases added by
hand). The key says which entities are one identity; the store says which share a parent. The score
is computed on the key's three books, and apart on the other 26. Entities the key does not name are
outside the score.

```
an entity of an Oz book matches a character when its name or an alias folds to the character's name
for each pair of matched entities in different documents:
    same character, same parent       -> united
    same character, different parents -> split
    different characters, same parent -> wrong join
recall = united / (united + split)
```

In [ ]:
def read_key(folder):
    """fold(name) -> character, over every character's names. A name two characters share is dropped."""
    key = json.loads((folder / "oz-key.json").read_text(encoding="utf-8"))
    owner = {}
    shared = set()
    for c in key["characters"]:
        for alias in c["aliases"]:
            f = fold(alias)
            if f in owner and owner[f] != c["name"]:
                shared.add(f)
            owner.setdefault(f, c["name"])
    for f in shared:
        owner.pop(f, None)
    return key, owner


def match_key(db, key, owner, unseen=False):
    """(books, the entities the key names with their characters, the entities it names twice). A node
    holding two characters' names is itself a join, so it is counted and never dropped in silence."""
    books = {}
    for doc_id, title, source_uri in db.execute("select doc_id, title, source_uri from document"):
        if "/oz/" in source_uri and (title in key["books"]) != unseen:
            books[doc_id] = title
    aliases = {}
    for doc_id, node_id, alias in db.execute("select doc_id, node_id, alias from alias"):
        if doc_id in books:
            aliases.setdefault((doc_id, node_id), []).append(alias)
    matched = {}
    twice = []
    for doc_id, node_id, name in db.execute("select doc_id, node_id, name from node"):
        if doc_id not in books or node_id.endswith(":doc"):
            continue
        hits = set()
        for form in [name] + aliases.get((doc_id, node_id), []):
            if fold(form) in owner:
                hits.add(owner[fold(form)])
        if len(hits) == 1:
            matched[(doc_id, node_id)] = hits.pop()
        elif hits:
            twice.append([name, books[doc_id], sorted(hits)])
    return books, matched, twice


def score_key(db, books, matched):
    """Cross-document pairs of one character united or split, and pairs of two characters joined."""
    parent = {}
    for doc_id, node_id, parent_id in db.execute("select doc_id, node_id, parent_id from instance_of"):
        parent[(doc_id, node_id)] = parent_id
    by_character = {}
    for child, character in matched.items():
        by_character.setdefault(character, []).append(child)
    rows = []
    united = 0
    split = 0
    for character, children in sorted(by_character.items()):
        documents = set()
        parents = set()
        for child in children:
            documents.add(child[0])
            parents.add(parent.get(child))
        same = 0
        apart = 0
        for i, a in enumerate(children):
            for b in children[i + 1:]:
                if a[0] != b[0]:
                    if parent.get(a) == parent.get(b):
                        same += 1
                    else:
                        apart += 1
        united += same
        split += apart
        rows.append((character, len(children), len(documents), len(parents), same, apart))
    joined = []
    children = sorted(matched)
    for i, a in enumerate(children):
        for b in children[i + 1:]:
            if matched[a] != matched[b] and parent.get(a) == parent.get(b) and parent.get(a) is not None:
                joined.append((matched[a], matched[b], books[a[0]], books[b[0]]))
    return rows, united, split, joined


def recall_of(united, split):
    if united + split == 0:
        return None
    return united / (united + split)


def report_key(store):
    """The score on the three books and, apart, on the other 26, written to oz-score.json."""
    key, owner = read_key(OZ_KEY)
    db = sqlite3.connect(store)
    books, matched, twice = match_key(db, key, owner)
    rows, united, split, joined = score_key(db, books, matched)
    later_books, later_matched, later_twice = match_key(db, key, owner, unseen=True)
    _, later_united, later_split, later_joined = score_key(db, later_books, later_matched)
    entities = 0
    for doc_id in list(books) + list(later_books):
        entities += count_of(db, f"select count(*) from node where doc_id = '{doc_id}' and node_id not like '%:doc'")
    db.close()
    covered = len(matched) + len(later_matched)
    print(f"the key covers {covered} of {len(books) + len(later_books)} books' {entities} entities ({covered / entities:.1%}); "
          f"{len(twice) + len(later_twice)} entities hold two characters' names")
    recall = recall_of(united, split)
    later_recall = recall_of(later_united, later_split)
    print(f"key: {len(key['characters'])} characters, revision {key['revision']}")
    print(f"{len(books)} books: {len(matched)} entities matched, {united} pairs united, {split} split, "
          f"recall {recall}, {len(joined)} wrong joins")
    print(f"{len(later_books)} unseen books: {len(later_matched)} entities matched, {later_united} pairs united, "
          f"{later_split} split, recall {later_recall}, {len(later_joined)} wrong joins")
    characters = []
    for c, n, d, p, u, s in rows:
        characters.append({"character": c, "children": n, "documents": d, "parents": p, "united": u, "split": s})
    wrong = []
    for row in joined:
        wrong.append(list(row))
    score = {"revision": key["revision"], "matched": len(matched), "united": united, "split": split, "recall": recall,
             "wrong_joins": len(joined),
             "unseen_books": {"books": len(later_books), "matched": len(later_matched), "united": later_united,
                              "split": later_split, "recall": later_recall, "wrong_joins": len(later_joined)},
             "entities_in_the_books": entities, "covered": covered, "names_two_characters": twice + later_twice,
             "characters": characters, "joined": wrong}
    (store.parent / "oz-score.json").write_text(json.dumps(score, ensure_ascii=False, indent=1), encoding="utf-8")


if __name__ == "__main__" and FINISHED and OZ_KEY:
    report_key(STORE)

## Block 17b: the joins to read

A sample of the parents that cross speakers is written out for a person to read, one parent to a
section with every line. The share read as wrong is reported beside the Oz score.

```
parents with instances from 2 or more speakers and 2 or more distinct sessions: draw JOINS_TO_READ
write each one's name, kind, speakers, sessions and lines to joins-to-read.md
```

In [ ]:
JOINS_TO_READ = 100


def speakers_of_parents(db):
    """parent -> (its speakers, its session files), over the parents held only by chats."""
    chats = set()
    for (doc_id,) in db.execute(CHATS):
        chats.add(doc_id)
    place = {}
    for doc_id, source_uri in db.execute("select doc_id, source_uri from document"):
        place[doc_id] = PurePosixPath(source_uri)
    held = {}
    for doc_id, parent_id in db.execute("select doc_id, parent_id from instance_of"):
        held.setdefault(parent_id, []).append(doc_id)
    spans = {}
    for parent_id, doc_ids in held.items():
        speakers = set()
        sessions = set()
        for doc_id in doc_ids:
            if doc_id in chats:
                speakers.add(place[doc_id].parent.name)
                sessions.add(place[doc_id].name)
        if len(speakers) >= 2 and len(sessions) >= 2:
            spans[parent_id] = (len(speakers), len(sessions))
    return spans


def write_joins(store):
    """A sample of the parents crossing speakers, for a person to read."""
    db = sqlite3.connect(store)
    spans = speakers_of_parents(db)
    chosen = random.Random(494).sample(sorted(spans), min(JOINS_TO_READ, len(spans)))
    parts = [f"# Parents crossing speakers: {len(chosen)} drawn of {len(spans)}\n"]
    for parent_id in chosen:
        name, kind, summary = db.execute("select name, kind, summary from parent where parent_id = ?", (parent_id,)).fetchone()
        speakers, sessions = spans[parent_id]
        parts.append(f"## {plain(name)} ({kind}): {speakers} speakers, {sessions} sessions\n\n{plain(without_tags(summary))}\n")
    db.close()
    (store.parent / "joins-to-read.md").write_text("\n".join(parts), encoding="utf-8")
    print(f"joins-to-read.md: {len(chosen)} of the {len(spans)} parents crossing speakers")


if __name__ == "__main__" and FINISHED:
    write_joins(STORE)

## Block 18: collections and topics

The bodies of work from Block 14 go into the store. Chat sessions are grouped by topic modelling.

**Chat sessions** are topics: BERTopic over the sessions' abstract vectors (Grootendorst 2022), with
k-means at root-N topics. The topics set the centroids: each session is then rebalanced to its nearest
centroid in the full embedding space, and one under TOPIC_FLOOR is Other. Past 16 topics, Ward linkage
over the centroids gives fourth-root-N parents, and only the leaves are rebalanced. A session's
abstract is embedded without the words "the user", which every one opens with. One call titles a
group from the five abstracts nearest its centre. Topics are built once, at the end.

The score at the end compares the bodies of work with the source folders, on all works and on those
of dumps 6 to 10. Each of the 20 novels is its own class, and the score counts the novels left alone.

```
save each body of 2 or more, and its subgroups

topics over the chat sessions:
    k = ceiling(sqrt(N))
    labels = BERTopic(the sessions' vectors, UMAP, k-means with k clusters)
    centroids = mean vector of each topic, in the full embedding space
    each session -> nearest centroid, or Other under .65
    if more than 16 topics:
        Ward over the centroids, down to ceiling(N^(1/4)) subjects
    one call titles each topic and subject

score: harmonic purity, ARI and NMI against the source folders
```

In [ ]:
COLLECTION = """Below are the chat sessions nearest the centre of one group of {n} sessions, found because their
abstracts read alike. Name the group by its subject, the way a reader would look for it. Then write an
abstract of two or three sentences saying what the sessions cover. Say only what the abstracts below
say; name nothing that is not in them.

Documents:
{documents}

Reply with a JSON object: {{"name": "<name>", "abstract": "<two or three sentences>"}}."""


def find_topics(texts, vectors):
    """BERTopic's topics with k-means at root-N; only their membership is used, for the centroids."""
    from bertopic import BERTopic
    from sklearn.cluster import KMeans
    from umap import UMAP
    reducer = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=494)
    clusters = KMeans(n_clusters=math.ceil(math.sqrt(len(texts))), random_state=494, n_init=10)
    model = BERTopic(umap_model=reducer, hdbscan_model=clusters, verbose=False)
    labels, _ = model.fit_transform(texts, embeddings=vectors)
    return labels


def rebalance(vectors, labels):
    """Each session to its nearest topic centroid at TOPIC_FLOOR, else Other. (groups of rows, centroids)."""
    labels = numpy.array(labels)
    topics = []
    for t in set(labels.tolist()):
        if t >= 0:
            topics.append(t)
    topics.sort()
    centres = []
    for t in topics:
        centres.append(vectors[labels == t].mean(axis=0))
    centres = numpy.stack(centres)
    centres = centres / numpy.linalg.norm(centres, axis=1, keepdims=True)
    sims = vectors @ centres.T
    nearest = sims.argmax(axis=1)
    groups = []
    for t in topics:
        groups.append([])
    for i, c in enumerate(nearest):
        if sims[i, c] >= TOPIC_FLOOR:
            groups[int(c)].append(i)
    kept_groups = []
    kept = []
    for c, group in enumerate(groups):
        if group:
            kept_groups.append(group)
            kept.append(c)
    return kept_groups, centres[kept]


def ward(centres, sizes, want):
    """Ward linkage over centroids weighted by size, down to `want` groups of topic numbers."""
    groups = []
    weights = []
    points = []
    for i in range(len(centres)):
        groups.append([i])
        weights.append(float(sizes[i]))
        points.append(centres[i].copy())
    while len(groups) > want:
        best = None
        pair = None
        for a in range(len(groups)):
            for b in range(a):
                gap = points[a] - points[b]
                cost = (weights[a] * weights[b] / (weights[a] + weights[b])) * float(gap @ gap)
                if best is None or cost < best:
                    best = cost
                    pair = (a, b)
        a, b = pair
        total = weights[a] + weights[b]
        points[b] = (points[a] * weights[a] + points[b] * weights[b]) / total
        weights[b] = total
        groups[b] += groups[a]
        del groups[a], weights[a], points[a]
    return groups


def near_centre(vectors, group):
    """The five rows of a group nearest its centroid, nearest first."""
    centre = vectors[group].mean(axis=0)
    order = numpy.argsort(-(vectors[group] @ centre))[:5]
    rows = []
    for i in order:
        rows.append(group[int(i)])
    return rows


def abstract_text(text):
    for words in ("The user", "the user", "They ", "they "):
        text = text.replace(words, "")
    return text


def title_group(doc_ids, near):
    """A group named and described from the abstracts nearest its centre."""
    entries = []
    for title, text in near:
        entries.append(f"- {title}: {text[:500]}")
    prompt = COLLECTION.format(n=len(doc_ids), documents="\n\n".join(entries))
    reply = generate(prompt, {"name": str, "abstract": str}, "collection")
    if reply:
        return reply["name"], reply["abstract"]
    return f"Works like {near[0][0]}", f"{len(doc_ids)} documents."


def save_groups(db, number, ids, vectors, groups, titles, abstracts, within):
    """One call for each group, then its rows. (the next free collection id, each group's id)."""
    jobs = []
    for group in groups:
        doc_ids = []
        for i in group:
            doc_ids.append(ids[i])
        near = []
        for i in near_centre(vectors, group):
            near.append((titles[ids[i]], abstracts[ids[i]]))
        jobs.append((doc_ids, near))
    named = in_parallel(title_group, jobs)
    numbers = []
    for (doc_ids, _), (name, abstract), parent in zip(jobs, named, within):
        db.execute("insert into collection values (?,?,?,?)", (number, name, abstract, parent))
        for doc_id in doc_ids:
            db.execute("insert into document_in values (?,?)", (doc_id, number))
        numbers.append(number)
        number += 1
    return number, numbers


def save_bodies(db, state):
    """Each body of two or more works, largest first, and its subgroups. The next free collection id."""
    ranked = []
    for group in state["groups"]:
        ranked.append((-len(group["works"]), group["number"]))
    ranked.sort()
    by_number = {}
    for group in state["groups"]:
        by_number[group["number"]] = group
    number = 0
    for size, group_number in ranked:
        group = by_number[group_number]
        if len(group["works"]) < 2:
            continue
        body = number
        db.execute("insert into collection values (?,?,?,?)", (body, group["name"], group["description"], None))
        for doc_id in group["works"]:
            db.execute("insert into document_in values (?,?)", (doc_id, body))
        number += 1
        for sub in group["subgroups"]:
            db.execute("insert into collection values (?,?,?,?)", (number, sub["name"], "", body))
            for doc_id in sub["works"]:
                db.execute("insert into document_in values (?,?)", (doc_id, number))
            number += 1
    return number


def read_documents(db):
    docs = {}
    for doc_id, title, occurred_at, source_uri in db.execute("select doc_id, title, occurred_at, source_uri from document"):
        docs[doc_id] = {"title": title, "occurred_at": occurred_at, "source_uri": source_uri}
    return docs


def build_topics(db, number, docs):
    """The chat sessions' topics, and subjects above them when there are many."""
    titles = {}
    for doc_id, doc in docs.items():
        titles[doc_id] = doc["title"]
    abstracts = read_abstracts(db)
    sessions = []
    for (doc_id,) in db.execute(CHATS):
        if abstracts.get(doc_id):
            sessions.append(doc_id)
    ids = by_date(sessions, docs)
    if not ids:
        return
    texts = []
    for doc_id in ids:
        texts.append(abstract_text(abstracts[doc_id]))
    vectors = embed(texts)
    groups, centres = rebalance(vectors, find_topics(texts, vectors))
    within = [None] * len(groups)
    if len(groups) > FLAT_TOPICS:
        sizes = []
        for group in groups:
            sizes.append(len(group))
        sets = ward(centres, sizes, math.ceil(len(ids) ** 0.25))
        joined = []
        for topics in sets:
            rows = []
            for t in topics:
                rows += groups[t]
            joined.append(rows)
        number, parent_ids = save_groups(db, number, ids, vectors, joined, titles, abstracts, [None] * len(joined))
        for topics, parent_id in zip(sets, parent_ids):
            for t in topics:
                within[t] = parent_id
    save_groups(db, number, ids, vectors, groups, titles, abstracts, within)
    placed = 0
    for group in groups:
        placed += len(group)
    print(f"chats: {len(groups)} topics over {len(ids)} sessions, {len(ids) - placed} in Other; ${SPENT:.2f}")


def build_collections(store):
    """The bodies of work and the chats' topics, written from scratch."""
    db = sqlite3.connect(store)
    db.execute("delete from collection")
    db.execute("delete from document_in")
    number = save_bodies(db, read_library())
    build_topics(db, number, read_documents(db))
    db.commit()
    db.close()


def purity(found, truth):
    """The share of items in the largest true class of their found group."""
    counts = {}
    for f, t in zip(found, truth):
        counts.setdefault(f, {})
        counts[f][t] = counts[f].get(t, 0) + 1
    right = 0
    for classes in counts.values():
        right += max(classes.values())
    return right / len(found)


def score_bodies(store, works=None):
    """Bodies of work against the source folders: harmonic purity, ARI, NMI, novels left alone."""
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    db = sqlite3.connect(store)
    docs = read_documents(db)
    db.close()
    state = read_library()
    label = {}
    size = {}
    for group in state["groups"]:
        size[group["number"]] = len(group["works"])
        for doc_id in group["works"]:
            label[doc_id] = group["number"]
    chosen = works
    if chosen is None:
        chosen = state["placed"]
    truth = []
    found = []
    novels = 0
    alone_novels = 0
    for doc_id in chosen:
        found.append(label[doc_id])
        if "graphrag-bench" in docs[doc_id]["source_uri"]:      # a novel is its own class
            truth.append(doc_id)
            novels += 1
            if size[label[doc_id]] == 1:
                alone_novels += 1
        else:
            truth.append(source_set(docs[doc_id]))
    forward = purity(found, truth)
    backward = purity(truth, found)
    return {"works": len(chosen), "harmonic_purity": 2 * forward * backward / (forward + backward),
            "ari": adjusted_rand_score(truth, found), "nmi": normalized_mutual_info_score(truth, found),
            "novels_left_alone": alone_novels, "novels": novels}


def score_library(store):
    """The score on all works, and on those of dumps 6 to 10."""
    db = sqlite3.connect(store)
    dumps = read_dumps(read_documents(db))
    db.close()
    placed = set(read_library()["placed"])
    held_out = []
    for dump in dumps[5:]:
        for doc_id in dump:
            if doc_id in placed:
                held_out.append(doc_id)
    print("all works:", score_bodies(store))
    print("dumps 6 to 10:", score_bodies(store, held_out))


if __name__ == "__main__" and FINISHED:
    build_collections(STORE)
    score_library(STORE)

## Block 19: the collection figure

The bodies of work as hubs on a ring and their documents fanned around each, written as an SVG beside
the store.

```
for each body: place a hub on a ring, fan its documents around it, draw the links
save collection-radial.svg
```

In [ ]:
def library_collections(db):
    """The bodies of work, largest first."""
    query = f"""select c.collection_id, count(*) as n from collection c
                join document_in i on i.collection_id = c.collection_id
                where c.within is null
                group by c.collection_id having sum(i.doc_id in ({CHATS})) = 0 order by n desc"""
    bodies = []
    for collection_id, n in db.execute(query):
        bodies.append(collection_id)
    return bodies


def draw_collection_radial(store, out):
    """The bodies of work on a ring, their documents fanned around each."""
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import pyplot
    db = sqlite3.connect(store)
    titles = dict(db.execute("select doc_id, title from document"))
    names = dict(db.execute("select collection_id, name from collection"))
    bodies = library_collections(db)
    members = {}
    for d, c in db.execute("select doc_id, collection_id from document_in"):
        members.setdefault(c, []).append(d)
    db.close()
    figure, axis = pyplot.subplots(figsize=(15, 15))
    for i, body in enumerate(bodies):
        angle = 2 * math.pi * i / len(bodies)
        hub_x = 10.0 * math.cos(angle)
        hub_y = 10.0 * math.sin(angle)
        works = members[body]
        for j, d in enumerate(works):
            a = angle + (j - (len(works) - 1) / 2) * (math.pi / max(len(works), 4))
            x = hub_x + 3.2 * math.cos(a)
            y = hub_y + 3.2 * math.sin(a)
            axis.plot([hub_x, x], [hub_y, y], color="#9aa5b1", linewidth=0.9, alpha=0.7, zorder=1)
            axis.scatter([x], [y], s=70, c="#7aa6c2", edgecolors="white", zorder=2)
            axis.annotate(titles[d][:30], (x, y), fontsize=7, xytext=(5, 3), textcoords="offset points", color="#23303f")
        axis.scatter([hub_x], [hub_y], s=700, c="#b4552a", edgecolors="white", linewidths=2, zorder=3)
        axis.annotate(names[body][:38], (hub_x, hub_y), fontsize=10, fontweight="bold", ha="center", xytext=(0, 16),
                      textcoords="offset points", color="#1f3d4a")
    axis.set_aspect("equal")
    axis.set_axis_off()
    axis.set_title("collections and their documents", color="#1f3d4a")
    figure.savefig(out / "collection-radial.svg", bbox_inches="tight", facecolor="#fbf7f0")
    pyplot.close(figure)
    print(f"collection-radial.svg: {len(bodies)} collections")


if __name__ == "__main__" and FINISHED:
    draw_collection_radial(STORE, OUT)